# SCF Phase 3 benchmark shard: **B_main**

Self-contained pass-2 runner. Regenerates the processed design from primary
sources, verifies its sha256 against the frozen value, then runs this
config's evaluation arms against the embedded calibration (pass-1 freeze).
Runtime target < 6 h on CPU; checkpoint-resume safe.

In [ ]:
%pip install -q numpy scipy pandas pyarrow matplotlib
import os
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[v] = '1'
print('env set')

### harness: `de_formulas.py` (sha256 5dffb441b6382d00)

In [ ]:
harness_de_formulas = "\"\"\"Deterministic-equivalent (DE) formula sheet implementations for SCF Phase 1.\n\nConventions (see docs/model_card.md):\n  - sigma_u^2 = 1 default; spike strengths l_j are eigenvalues of Lambda Lambda' / sigma_u^2.\n  - Population eigenvalues of Sigma_X / sigma_u^2 are tau_j = 1 + l_j.\n  - c = p/n.\n  - gamma is given in factor coordinates (gamma_j multiplies population eigenvector u_j).\n\nEvery function here is pure numpy/scipy and carries its derivation in the\ndocstring. Each transcribed classical formula has a numerical self-check in\ntests/test_identities.py before it is used anywhere (research plan WP 1.3).\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport numpy as np\n\n# ---------------------------------------------------------------------------\n# Classical random matrix facts (transcribed; self-checked in tests)\n# ---------------------------------------------------------------------------\n\n\ndef mp_edges(c: float, sigma2: float = 1.0) -> tuple[float, float]:\n    \"\"\"Marchenko-Pastur bulk edges for sample covariance of white data.\n\n    F6. For X (n x p) with iid columns of variance sigma2, the eigenvalues of\n    X'X/n lie in [sigma2 (1-sqrt(c))^2, sigma2 (1+sqrt(c))^2] asymptotically,\n    c = p/n. Source: Marchenko-Pastur (1967); Bai-Silverstein (2010) Ch. 3.\n    \"\"\"\n    s = np.sqrt(sigma2)\n    return (s * (1 - np.sqrt(c)) ** 2, s * (1 + np.sqrt(c)) ** 2)\n\n\ndef bbp_location(l: float, c: float, sigma2: float = 1.0) -> float:\n    \"\"\"BBP sample-eigenvalue location for a population spike of strength l.\n\n    F4. Population eigenvalue tau = sigma2 (1 + l). For l > sqrt(c) the\n    corresponding sample eigenvalue converges a.s. to\n\n        mu(l) = tau (1 + c sigma2 / (tau - sigma2))\n              = sigma2 (1 + l) (l + c) / l,\n\n    and to the bulk edge (1+sqrt(c))^2 otherwise.\n\n    CORRECTION vs research plan Section 2.3 item 4, which wrote\n    mu(l) = (1+l)(1+cl)/l. The two agree only when l = 1 or c = 1\n    (since 1 + cl = l + c iff (l-1)(c-1) = 0). The correct BBP form is\n    (1+l)(1 + c/l); verified numerically in tests (test_bbp_location).\n    Source: Baik-Ben Arous-Peche (2005), Thm 2.1 (real case); Johnstone (2001).\n    \"\"\"\n    if l <= np.sqrt(c):\n        return sigma2 * (1 + np.sqrt(c)) ** 2\n    return sigma2 * (1 + l) * (l + c) / l\n\n\ndef bgn_overlap(l: float, c: float) -> float:\n    \"\"\"Squared eigenvector overlap |<u_j, v_hat_j>|^2 for a spike of strength l.\n\n    F5. For l > sqrt(c): xi(l, c) = (1 - c/l^2) / (1 + c/l); else 0.\n    Source: Benaych-Georges-Nadakuditi (2011), Adv. Math. 227(1), real case;\n    also (2012) for the rectangular/singular-vector version.\n    \"\"\"\n    if l <= np.sqrt(c):\n        return 0.0\n    return (1 - c / l**2) / (1 + c / l)\n\n\ndef tw_mu_sigma(n: int, p: int) -> tuple[float, float]:\n    \"\"\"Johnstone center/scale for the largest eigenvalue of X'X (unscaled).\n\n    F7. For white X (n x p), entries N(0, sigma2), the largest eigenvalue of\n    X'X (NOT divided by n) satisfies\n\n        (lam_max - mu_np) / sigma_np  ->  TW1,\n\n    mu_np   = (sqrt(n-1) + sqrt(p))^2,\n    sigma_np = (sqrt(n-1) + sqrt(p)) * (1/sqrt(n-1) + 1/sqrt(p))^{1/3},\n\n    multiplied by sigma2. Source: Johnstone (2001), Ann. Statist. 29(2).\n    \"\"\"\n    sn, sp = np.sqrt(n - 1.0), np.sqrt(float(p))\n    mu = (sn + sp) ** 2\n    sigma = (sn + sp) * (1.0 / sn + 1.0 / sp) ** (1.0 / 3.0)\n    return mu, sigma\n\n\n# Tracy-Widom order-1 quantiles (standard published tables, e.g. Johnstone\n# 2001 Table 1 / Edelman; used only as thresholds, calibrated by MC in tests).\nTW1_Q95 = 0.9793\nTW1_Q99 = 2.0234\n\n\ndef tw_threshold(n: int, p: int, sigma2: float = 1.0, q: float = TW1_Q99) -> float:\n    \"\"\"Upper-q threshold for lambda_max of X'X/n (white null), cov scale.\n\n    F7. threshold = sigma2 * (mu_np + q * sigma_np) / n.\n    \"\"\"\n    mu, sigma = tw_mu_sigma(n, p)\n    return sigma2 * (mu + q * sigma) / n\n\n\n# ---------------------------------------------------------------------------\n# Bias functionals (M1). gamma given in factor coordinates; Q columns are the\n# population eigenvectors u_j.\n# ---------------------------------------------------------------------------\n\n\ndef sigma_x_eigs(l: np.ndarray, sigma2: float = 1.0) -> np.ndarray:\n    \"\"\"Eigenvalues of Sigma_X = sigma2 I + Lambda Lambda' (F1).\"\"\"\n    l = np.asarray(l, float)\n    return sigma2 * (1.0 + l)\n\n\ndef ols_bias_vector(l: np.ndarray, gamma: np.ndarray, sigma2: float = 1.0) -> np.ndarray:\n    \"\"\"Population OLS bias b_ols = Sigma_X^{-1} Lambda gamma (F1).\n\n    Identity: Cov(X, Y) = Sigma_X beta + Lambda gamma with\n    Sigma_X = sigma2 I + Lambda Lambda'. Since Lambda gamma =\n    sum_j sigma_u sqrt(l_j) gamma_j u_j,\n\n        plim beta_OLS - beta = sum_j [sqrt(l_j / sigma2) / (1 + l_j)] gamma_j u_j.\n\n    EXACT finite-n fact (c <= 1): writing w = Lambda f + eps = X a + zeta with\n    a = Sigma_X^{-1} Lambda gamma and zeta independent of X (joint Gaussianity,\n    A2/A3), E[beta_OLS] - beta = a exactly, because (X'X)^{-1} X' X = I and\n    E[(X'X)^{-1} X' zeta] = E[(X'X)^{-1} X'] E[zeta] = 0. So the simulated mean\n    bias should equal this vector up to MC error at any n with p <= n.\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    return np.sqrt(l / sigma2) / (1.0 + l) * gamma\n\n\ndef ridge_bias_vector(\n    l: np.ndarray, gamma: np.ndarray, lam: float, sigma2: float = 1.0\n) -> np.ndarray:\n    \"\"\"Population ridge bias (F2): (Sigma_X + lam I)^{-1} Lambda gamma.\n\n    plim beta_ridge - beta\n        = sum_j [sigma_u sqrt(l_j) / (sigma_u^2 (1 + l_j) + lam)] gamma_j u_j,\n    i.e., component form sqrt(l_j sigma2) / (sigma2 (1 + l_j) + lam) * gamma_j.\n    Finite-n deviation order 1/n (shrinkage fluctuation of (Sigmahat+lam)^{-1}).\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    return np.sqrt(l * sigma2) / (sigma2 * (1.0 + l) + lam) * gamma\n\n\ndef pca_trim_bias_vector(\n    l: np.ndarray, gamma: np.ndarray, k: int, sigma2: float = 1.0\n) -> np.ndarray:\n    \"\"\"Population PCA-k trim bias (F3).\n\n    Regressing Y on the top-k population PCs gives\n        beta_trim = P_k beta + V_k (V_k' Sigma V_k)^{-1} V_k' Lambda gamma,\n    so the bias keeps only the retained directions:\n        bias = sum_{j <= k} [sqrt(l_j sigma2)/(sigma2(1+l_j))] gamma_j u_j.\n    Trimming removes the bias of dropped components entirely and leaves the\n    retained components untouched; the cost is signal loss\n    1 - ||P_k beta||^2 / ||beta||^2 (large under dense beta, A4a).\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    out = np.zeros_like(gamma)\n    out[:k] = np.sqrt(l[:k] * sigma2) / (sigma2 * (1.0 + l[:k])) * gamma[:k]\n    return out\n\n\ndef mp_stieltjes_inv(lam: float, a: float) -> float:\n    \"\"\"E[1/(t + lam)] under the Marchenko-Pastur law with parameter a <= 1.\n\n    F8 helper. For the nonzero spectrum of an n x n Gram matrix built from p\n    columns (aspect a = n/p <= 1), the Stieltjes transform at z = -lam solves\n    lam m^2 + (lam + a) m - 1 = 0, giving\n\n        m_inv(lam) = (sqrt((lam+a)^2 + 4 lam) - (lam + a)) / (2 lam)  > 0,\n\n    and E[1/(t+lam)] = m_inv. Source: standard MP Stieltjes equation\n    (Bai-Silverstein 2010, Ch. 3); consistency checked numerically in tests.\n    \"\"\"\n    disc = np.sqrt((lam + a) ** 2 + 4.0 * lam)\n    return (disc - (lam + a)) / (2.0 * lam)\n\n\ndef ridge_capture(l: np.ndarray, lam: float, c: float, sigma2: float = 1.0) -> np.ndarray:\n    \"\"\"Capture coefficients of E[(Sigmahat + lam)^{-1} Sigmahat] on spike dirs, c>1.\n\n    F8 companion. On the spike direction u_j the sample spike sits at nu_j =\n    bbp_location(l_j, c) and contributes xi_j * nu_j / (nu_j + lam); the leaked\n    mass (1 - xi_j) sees the bulk operator, whose ridge-weighted trace gives\n    (1/c) * E[t/(t+lam)] = (1/c)(1 - lam * m_inv(lam, a=1/c)):\n\n        cap_j(lam) = xi_j nu_j/(nu_j + lam)\n                     + (1 - xi_j) (1/c) (1 - lam * m_inv(lam, 1/c)).\n\n    Consistency anchor: lim_{lam->0} cap_j = xi_j + (1-xi_j)/c, i.e., the\n    SUPERSEDED xi-based min-norm guess (see bgn_capture_superseded). The\n    lambda-dependence itself is calibrated empirically by the WP 1.5 revisit\n    cells before Phase 2 uses it; treat as PROVISIONAL for c > 1.\n    \"\"\"\n    l = np.asarray(l, float)\n    xi = np.array([bgn_overlap(li, c) for li in l])\n    nu = np.array([bbp_location(li, c, sigma2) for li in l])\n    a = 1.0 / c\n    bulk = (1.0 - lam * mp_stieltjes_inv(lam, a)) / c\n    return xi * nu / (nu + lam) + (1.0 - xi) * bulk\n\n\ndef minnorm_capture(l: np.ndarray, c: float) -> np.ndarray:\n    \"\"\"Diagonal capture coefficients of E[P_row] on spike directions, c > 1 (F8).\n\n    PILOT-VALIDATED CONJECTURE (Phase 1, WP 1.5): the capture law\n\n        cap_j = (1 + l_j) / (c + l_j)\n\n    matched simulation to ~0.5-1 percent at c = 5 across supercritical\n    (l = 3 sqrt(c)) and subcritical (l = 0.5 sqrt(c), l ~ 0) components and\n    both n = 400 and n = 2000; see docs/de_formula_sheet.md F8. Boundary\n    anchors: l -> 0 gives the uniform rowspace fraction 1/c; l -> inf gives 1\n    (consistent estimation of spike-aligned components). For c <= 1 OLS is\n    full rank and capture is identically 1.\n\n    The earlier xi-based formula xi(l,c) + (1 - xi(l,c))/c predicted 0.607 at\n    (l, c) = (3 sqrt(5), 5) against a measured 0.657-0.659 and is recorded in\n    bgn_capture_superseded for comparison only.\n    \"\"\"\n    l = np.asarray(l, float)\n    return (1.0 + l) / (c + l)\n\n\ndef bgn_capture_superseded(l: np.ndarray, c: float) -> np.ndarray:\n    \"\"\"Superseded xi-based capture guess (kept for audit trail).\"\"\"\n    l = np.asarray(l, float)\n    xi = np.array([bgn_overlap(li, c) for li in l])\n    return xi + (1.0 - xi) / c\n\n\ndef minnorm_bias_vector(\n    l: np.ndarray,\n    gamma: np.ndarray,\n    c: float,\n    beta_spike: np.ndarray | None = None,\n    sigma2: float = 1.0,\n) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Min-norm OLS bias decomposition for c > 1 (F8).\n\n    Returns (bias_confounding, bias_total_extra_terms) where\n        bias_confounding = sum_j capture_j * [sigma2 l_j/(sigma2(1+l_j))] gamma_j u_j,\n        total bias adds the fit artifact\n            sum_j (capture_j - 1) beta_j u_j + (1/c - 1) beta_perp\n    with beta_spike the coordinates of beta on (u_j) (dense A4a beta:\n    beta_j ~ N(0, 1/p), beta_perp norm^2 ~ 1 - r/p).\n\n    Derivation: see minnorm_capture docstring; the zeta decomposition makes\n    the confounding part exact at the DE level (only xi(l,c) itself is an\n    asymptotic object). Validated against simulation in the pilot (c = 5\n    overlay); flagged PROVISIONAL-NEW derivation in de_formula_sheet.md.\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    cap = minnorm_capture(l, c)\n    bias_conf = cap * ols_bias_vector(l, gamma, sigma2)\n    if beta_spike is None:\n        return bias_conf, np.zeros_like(bias_conf)\n    beta_spike = np.asarray(beta_spike, float)\n    fit_spike = (cap - 1.0) * beta_spike\n    return bias_conf, fit_spike\n\n\ndef minnorm_total_bias_norm(\n    l: np.ndarray,\n    gamma: np.ndarray,\n    c: float,\n    p: int,\n    sigma2: float = 1.0,\n) -> float:\n    \"\"\"RMS-over-beta prediction of ||E[beta_hat] - beta|| for c > 1 (F8).\n\n    Under A4a, beta_j ~ N(0, 1/p)-scale coordinates and\n    ||beta_perp||^2 ~ 1 - r/p, so\n\n        E||bias||^2 = sum_j [(cap_j - 1)^2 / p + (cap_j * ols_j * gamma_j)^2]\n                      + (1/c - 1)^2 (1 - r/p).\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    r = len(l)\n    cap = minnorm_capture(l, c)\n    ols = ols_bias_vector(l, gamma, sigma2)\n    e2 = np.sum((cap - 1.0) ** 2 / p + (cap * ols * gamma) ** 2)\n    e2 += (1.0 / c - 1.0) ** 2 * max(0.0, 1.0 - r / p)\n    return float(np.sqrt(e2))\n\n\n# ---------------------------------------------------------------------------\n# Detection quantities (C2 scaffolding; frontier calibration is Phase 2)\n# ---------------------------------------------------------------------------\n\n\ndef eff_detect_spike(\n    l: np.ndarray, gamma: np.ndarray, c: float, sigma2: float = 1.0\n) -> float:\n    \"\"\"Effective detection spike s_eff (F9, working definition).\n\n    s_eff = || P_spike Sigma_X^{-1/2} Lambda gamma ||^2\n          = sum_{j: l_j > sqrt(c)} [sigma2 l_j / (sigma2 (1 + l_j))] gamma_j^2.\n\n    Only the supercritical-aligned part of b = Lambda gamma produces a coherent\n    rank-r mean shift in the whitened cross-moment statistic S1; subcritical\n    components hide in the bulk (s_eff = 0 at leading order). This is the\n    formal content of \"invisible\" in the decoupling claim. The mapping from\n    s_eff to power (the frontier curve) is calibrated in Phase 2 (OMH\n    template); treated as heuristic in Phase 1.\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    sup = l > np.sqrt(c)\n    return float(np.sum((sigma2 * l[sup] / (sigma2 * (1.0 + l[sup]))) * gamma[sup] ** 2))\n\n\n# ---------------------------------------------------------------------------\n# Phase 2 detection formulas: F12 (max-z law) and F13 (augmented BBP).\n# Frozen spec: docs/detection_statistics.md; preregistration:\n# docs/phase2_preregistration.md. NEW-DE items validated by WP 2.1 null cells\n# before powering WP 2.3 gates.\n# ---------------------------------------------------------------------------\n\n\ndef maxz_null_var(\n    d: float,\n    c: float,\n    sigma_eps2: float = 1.0,\n    sigma_y2: float | None = None,\n) -> float:\n    \"\"\"Null variance of the standardized spike coordinate z_j (F12, ERRATUM 1).\n\n    With Y standardized, b = X_c' Ytilde / n, w_j = v_j' b and\n    z_j = sqrt(n) w_j / sqrt(d_j), under H0 (gamma = 0), A4a, A2/A3:\n\n        Var(w_j) = d_j^2 / p + sigma_eps2 d_j / (n sigma_y2^2),\n        =>  Var(z_j) = [ d_j / c + sigma_eps2 ] / sigma_y2^2,\n\n    where sigma_y2 = beta' Sigma beta + sigma_eps2 (approximately\n    tr(Sigma)/p + sigma_eps2 under A4a) enters because Y is standardized.\n    ERRATUM: an earlier draft claimed Var(z_j) = 1 + c d_j; that inverted the\n    rowspace factor (n d_j / p = d_j / c, not c d_j). Caught by\n    tests/test_phase2.py::test_maxz_null_variance BEFORE any Phase 2 sweep\n    data was generated (2026-08-24); see docs/detection_statistics.md\n    Erratum 1. Practical reading: dense-beta leakage through spike directions\n    GROWS like d_j/c, so calibrated spike-coordinate tests pay a large tax at\n    small c; the empirical detection frontier must be read jointly with S1.\n    \"\"\"\n    base = d / c + sigma_eps2\n    return base / sigma_y2 ** 2 if sigma_y2 else base\n\n\ndef mp_quantile(q: float, c: float) -> float:\n    \"\"\"q-th quantile of the Marchenko-Pastur law (ratio c <= 1 branch).\n\n    Density f(x) = sqrt((b - x)(x - a)) / (2 pi c x) on [a, b] with\n    a = (1 - sqrt(c))^2, b = (1 + sqrt(c))^2. Bisection on the explicit CDF;\n    used to convert a bulk-eigenvalue median into a sigma_u^2 estimate.\n    \"\"\"\n    a, b = (1 - np.sqrt(c)) ** 2, (1 + np.sqrt(c)) ** 2\n\n    def cdf(x):\n        # numeric CDF via fine trapezoid on the explicit density\n        xs = np.linspace(a, x, 4001)\n        dens = np.sqrt(np.maximum((b - xs) * (xs - a), 0.0)) / (2 * np.pi * c * xs)\n        return float(np.trapezoid(dens, xs))\n\n    lo, hi = a, b\n    for _ in range(80):\n        mid = 0.5 * (lo + hi)\n        if cdf(mid) < q:\n            lo = mid\n        else:\n            hi = mid\n    return 0.5 * (lo + hi)\n\n\ndef maxz_threshold(\n    ktop: int, alpha: float = 0.05, two_sided: bool = True\n) -> float:\n    \"\"\"Bonferroni Gaussian threshold for the max over ktop calibrated z's.\"\"\"\n    from scipy.stats import norm\n\n    tail = alpha / (2 * ktop) if two_sided else alpha / ktop\n    return float(norm.ppf(1.0 - tail))\n\n\ndef bbp_invert(nu: float, c: float) -> float:\n    \"\"\"Population spike l with sample location nu (F4 inverse).\n\n    Solves (1 + l)(l + c)/l = nu for l > sqrt(c): l^2 + (1 + c - nu) l + c = 0,\n    larger root. Returns np.nan if nu <= bulk edge (no supercritical spike can\n    sit there). Used by the SEB tuner to map sample spikes back to l_j.\n    \"\"\"\n    if nu <= (1 + np.sqrt(c)) ** 2:\n        return np.nan\n    b = 1.0 + c - nu\n    disc = b * b - 4.0 * c\n    if disc < 0:\n        return np.nan\n    return 0.5 * (-b + np.sqrt(disc))\n\n\ndef estimate_noise_scales(d_desc: np.ndarray, c: float):\n    \"\"\"Frozen observable-only estimates (se2_hat, sigma_y2_hat).\n\n    se2_hat: sigma_u^2 proxy from the non-spike bulk median matched against\n    the MP median (bulk median = sigma_u2 * MP_median(min(c,1))); capped at\n    the observed bulk median. sigma_y2_hat = mean(d) + se2_hat\n    (tr(Sigmahat)/p + sigma_eps2 under A4a). Used identically by the\n    detection statistics and the SEB tuner so their calibrations agree.\n    \"\"\"\n    d_desc = np.asarray(d_desc, float)\n    edge = (1 + np.sqrt(c)) ** 2 * 1.05 if c <= 1 else np.inf\n    tail = d_desc[d_desc < edge] if c <= 1 else d_desc\n    med_bulk = float(np.median(tail)) if len(tail) else float(np.median(d_desc))\n    se2_hat = med_bulk / mp_quantile(0.5, min(c, 1.0))\n    se2_hat = float(min(se2_hat, med_bulk))\n    sigma_y2 = float(np.mean(d_desc)) + se2_hat\n    return se2_hat, sigma_y2\n\n\ndef aug_secular_term(lam: float, eigs_desc: np.ndarray) -> float:\n    \"\"\"sum_j tau_j^2 / (lam - tau_j) evaluated above the top eigenvalue.\n\n    F13 helper: the resolvent trace entering the augmented secular equation.\n    Caller guarantees lam > max(eigs); poles below lam are impossible then.\n    \"\"\"\n    t = np.asarray(eigs_desc, float)\n    return float(np.sum(t ** 2 / (lam - t)))\n\n\ndef aug_null_root(\n    eigs_desc: np.ndarray, sigma_y2: float, p_eff: int | None = None\n) -> float:\n    \"\"\"H0 location of lambda_max of the augmented moment matrix (F13).\n\n    Under H0 the cross-moment vector b = Sigma beta / sigma_y has isotropic\n    geometry (A4a), so E[b'(lam I - Sigma)^{-1} b]\n      = sum_j tau_j^2 / (p sigma_y^2 (lam - tau_j)),\n    and the out-of-bulk root solves\n\n        lam = 1 + aug_secular_term(lam) / (p sigma_y2).\n\n    Plug-in version uses estimated eigenvalues (descending, covariance scale)\n    and sigma_y2_hat; see detection.py for the estimator. Bisection on\n    [t_max + eps, upper]; the upper bracket comes from the conservative bound\n    lam < 1 + S/(lam - t_max) solved as a quadratic.\n    \"\"\"\n    t = np.sort(np.asarray(eigs_desc, float))[::-1]\n    p = len(t)\n    tmax = t[0]\n    lo = tmax * (1.0 + 1e-9) + 1e-12\n\n    def f(lam):\n        return lam - 1.0 - aug_secular_term(lam, t) / (p * sigma_y2)\n\n    # conservative upper bracket: replace every tau by tmax in denominators\n    s_total = float(np.sum(t ** 2))\n    # lam - 1 = S/(p sy2 (lam - tmax)) -> lam^2 - (1+tmax) lam + (tmax + S/(p sy2))\n    disc = (1.0 + tmax) ** 2 - 4.0 * (tmax + s_total / (p * sigma_y2))\n    hi = 0.5 * ((1.0 + tmax) + np.sqrt(max(disc, 1e-300))) * (1.0 + 1e-6) + 1e-6\n    flo, fhi = f(lo), f(hi)\n    if flo > 0:  # root coincides with the spike itself (degenerate); clamp\n        return lo\n    for _ in range(200):\n        mid = 0.5 * (lo + hi)\n        if f(mid) > 0:\n            hi = mid\n        else:\n            lo = mid\n        if hi - lo < 1e-10 * max(1.0, hi):\n            break\n    return 0.5 * (lo + hi)\n\n\ndef aug_h1_root(\n    l: np.ndarray,\n    gamma: np.ndarray,\n    sigma_y2: float,\n    sigma_u2: float = 1.0,\n) -> float:\n    \"\"\"H1 location prediction of the augmented statistic (F13).\n\n    Population-level secular equation with confounding:\n\n        lam = 1 + [ sum_j tau_j^2 / (p sigma_y2 (lam - tau_j)) ]\n                + [ sigma_u2 sum_j l_j gamma_j^2 / (sigma_y2 (lam - tau_j)) ],\n\n    derived from E[b] = (Sigma beta + Lambda gamma)/sigma_y: the A4a beta part\n    spreads isotropically while the gamma part contributes the coherent\n    rank-r term. The O(1/sqrt(p)) cross term between beta and gamma is\n    dropped (documented approximation, checked in WP 2.1 overlays).\n    \"\"\"\n    l = np.asarray(l, float)\n    gamma = np.asarray(gamma, float)\n    tau = sigma_u2 * (1.0 + l)\n    p = len(tau)\n\n    def f(lam):\n        beta_part = aug_secular_term(lam, tau) / (p * sigma_y2)\n        conf_part = sigma_u2 * float(\n            np.sum(l * gamma ** 2 / (lam - tau))\n        ) / sigma_y2\n        return lam - 1.0 - beta_part - conf_part\n\n    tmax = float(tau.max())\n    lo = tmax * (1.0 + 1e-9) + 1e-12\n\n    def g(lam):\n        return lam - 1.0 - aug_secular_term(lam, tau) / (p * sigma_y2)\n\n    s_total = float(np.sum(tau ** 2)) + p * sigma_u2 * float(np.sum(l * gamma ** 2))\n    disc = (1.0 + tmax) ** 2 - 4.0 * (tmax + s_total / (p * sigma_y2))\n    hi = 0.5 * ((1.0 + tmax) + np.sqrt(max(disc, 1e-300))) * (1.0 + 1e-6) + 1e-6\n    if f(lo) > 0 or g(hi) <= 0:\n        return lo\n    for _ in range(200):\n        mid = 0.5 * (lo + hi)\n        if f(mid) > 0:\n            hi = mid\n        else:\n            lo = mid\n        if hi - lo < 1e-10 * max(1.0, hi):\n            break\n    return 0.5 * (lo + hi)\n\n\ndef tw_width_cov_scale(n: int, p: int, sigma2: float = 1.0) -> float:\n    \"\"\"Johnstone TW width on the covariance scale for an (n, p) white design.\n\n    Used as the v1 fluctuation scale of T_aug around its secular root with\n    dims (n, p+1). Known to be only approximately right for spiked\n    deformations; the MC-calibrated threshold variant is co-recorded per\n    docs/detection_statistics.md.\n    \"\"\"\n    _, sig = tw_mu_sigma(n, p)\n    return sigma2 * sig / n\n\n\n# ---------------------------------------------------------------------------\n# Factor-number selection rules (baselines; constants flagged approximate)\n# ---------------------------------------------------------------------------\n\n# Onatski (2010) ED-ratio asymptotic critical values, k = 1..10 (his Table 1,\n# standard normal quantile-based construction). APPROXIMATE transcription;\n# used only for PCA-k baseline selection, refined in Phase 2.\nONATSKI_CRIT = np.array([2.19, 2.09, 2.04, 2.01, 1.99, 1.97, 1.96, 1.95, 1.94, 1.93])\n\n\ndef onatski_select(eigs_desc: np.ndarray, kmax: int = 10) -> int:\n    \"\"\"Onatski ratio rule: largest k with eigs[k-1]/eigs[k] > crit[k-1] (F10).\"\"\"\n    eigs_desc = np.asarray(eigs_desc, float)\n    kmax = min(kmax, len(eigs_desc) - 1, len(ONATSKI_CRIT))\n    k = 0\n    for j in range(kmax):\n        if eigs_desc[j] <= 0 or eigs_desc[j + 1] <= 0:\n            break\n        if eigs_desc[j] / eigs_desc[j + 1] > ONATSKI_CRIT[j]:\n            k = j + 1\n        else:\n            break\n    return max(k, 0)\n\n\ndef bai_ng_select(eigs_desc: np.ndarray, n: int, p: int, kmax: int = 10) -> int:\n    \"\"\"Bai-Ng PC-style selector (F11). PROVISIONAL approximation.\n\n    khat = argmin_k [ sum_{j>k} eigs_j + k * mean(eigs) * log(max(n,p)) * (n+p)/(np) ].\n    The exact Bai-Ng (2002) penalty constants are not load-bearing for Phase 1\n    (baseline selection only); flagged for exact transcription before Phase 2.\n    \"\"\"\n    eigs_desc = np.asarray(eigs_desc, float)\n    kmax = min(kmax, len(eigs_desc) - 1)\n    scale = float(np.mean(eigs_desc)) * np.log(max(n, p)) * (n + p) / (n * p)\n    vals = [np.sum(eigs_desc[k:]) + k * scale for k in range(0, kmax + 1)]\n    return int(np.argmin(vals))\n\n\n# ---------------------------------------------------------------------------\n# Ledger hashing (mechanical verification hook, WP 1.1)\n# ---------------------------------------------------------------------------\n\n\n# ---------------------------------------------------------------------------\n# Spectral weight families and SEB tuner targets (WP 2.2; frozen in\n# docs/phase2_preregistration.md)\n# ---------------------------------------------------------------------------\n\n\ndef trim_weights(d: np.ndarray, tau: float) -> np.ndarray:\n    \"\"\"Cevid et al. Trim transform on covariance eigenvalues: w = min(1, tau/d).\n\n    Their eq. (3.3) caps SINGULAR values at tau; on the covariance scale\n    d_cov = d_svd^2/n this is w_j = min(1, (tau_svd/svd_j)^2). We parameterize\n    directly on the covariance scale with tau_cov; default tuning\n    tau = median(d) corresponds exactly to their tau = median singular value.\n    \"\"\"\n    d = np.asarray(d, float)\n    return np.minimum(1.0, tau / np.maximum(d, 1e-300))\n\n\ndef lava_weights(d: np.ndarray, rho: float) -> np.ndarray:\n    \"\"\"LAVA / SDBoost spectral-loss weights w_j = 1/(1 + rho d_j).\n\n    Nava et al. eq. (14): w_i = n lam2/(n lam2 + s_i^2) with s_i singular\n    values; on the covariance scale s_i^2 = n d_i so w_i = 1/(1 + d_i/rho')\n    with n lam2 = rho'. We absorb constants into rho >= 0.\n    \"\"\"\n    d = np.asarray(d, float)\n    return 1.0 / (1.0 + rho * d)\n\n\ndef sdboost_path_coefficients(\n    z: np.ndarray, d: np.ndarray, w: np.ndarray, m: int, nu: float\n) -> np.ndarray:\n    \"\"\"SDBoost linear-base-learner path alpha_j(m) = (z_j/d_j)(1-(1-nu w_j)^m).\n\n    Direct consequence of their boosting recursion (Nava et al. Section 3):\n    coordinate j of the coefficient vector in the right-singular basis moves\n    toward its OLS value z_j/d_j at per-direction rate nu w_j. m -> inf\n    recovers min-norm OLS; early stopping is what creates deconfounding.\n    \"\"\"\n    d = np.maximum(np.asarray(d, float), 1e-300)\n    return (np.asarray(z, float) / d) * (\n        1.0 - (1.0 - nu * np.asarray(w, float)) ** int(m)\n    )\n\n\ndef seb_predicted_mse(\n    l_hat: np.ndarray,\n    g2_hat: np.ndarray,\n    d: np.ndarray,\n    tau: float,\n    c: float,\n    n: int,\n    sigma_eps2: float,\n    sigma_u2: float = 1.0,\n) -> float:\n    \"\"\"DE-predicted causal MSE of the soft-trim estimator at threshold tau.\n\n    SEB tuner objective (preregistration WP 2.2): for weights\n    w_j = min(1, tau/d_j),\n\n        MSE_pred(tau) = sum_j [ w_j cap_j sqrt(l_j sigma2)/(sigma2(1+l_j)) ghat_j ]^2\n                        + sum_j w_j^2 sigma_eps2 / (n d_j)\n                        + sum_j (1 - w_j)^2 / p          (A4a signal loss)\n                        + artifact floor (c > 1 rowspace proxy),\n\n    where cap_j = minnorm_capture(l_j, c) at c > 1 and 1 otherwise, and\n    l_hat/g2_hat are plug-in estimates from the spectrum and cross-moment\n    coordinates. Pure function of observables when fed estimates; the ORACLE\n    variant feeds the true (l, gamma^2) into the SAME objective (frozen\n    definition of eb_oracle_tau, preregistration).\n    \"\"\"\n    l = np.asarray(l_hat, float)\n    g2 = np.asarray(g2_hat, float)\n    d = np.asarray(d, float)\n    w = trim_weights(d[: len(l)], tau)\n    cap = minnorm_capture(l, c) if c > 1 else np.ones_like(l)\n    coef = np.sqrt(l * sigma_u2) / (sigma_u2 * (1.0 + l))\n    bias2 = float(np.sum((w * cap * coef * np.sqrt(g2)) ** 2))\n    var_term = float(\n        np.sum((w ** 2) * sigma_eps2 / (n * np.maximum(d[: len(l)], 1e-12)))\n    )\n    # A4a signal-attenuation cost: dense beta has ~1/p mass per direction,\n    # systematic shrinkage (1 - w_j) contributes sum_j (1-w_j)^2 / p.\n    atten = float(np.sum((1.0 - w) ** 2)) / max(c * n, 1)\n    artifact = max(0.0, 1.0 / c - 1.0) ** 2 * float(np.mean(w)) if c > 1 else 0.0\n    return bias2 + var_term + atten + artifact\n\n\ndef ucm_strength(eigs_desc: np.ndarray, n: int, p: int) -> float:\n    \"\"\"Rendsburg-et-al.-style confounding strength proxy (B2 baseline).\n\n    Documented approximation of the UCM point estimate: the fraction of total\n    variance carried by super-BBP-outlier directions relative to the white\n    expectation. Uses only the spectrum; bootstrap thresholding is applied by\n    the caller. NOT a faithful reimplementation of their full PE algorithm;\n    flagged APPROXIMATE in docs/detection_statistics.md.\n    \"\"\"\n    eigs_desc = np.asarray(eigs_desc, float)\n    edge = (1.0 + np.sqrt(p / n)) ** 2\n    excess = np.sum(np.maximum(eigs_desc - edge, 0.0))\n    return float(excess / max(np.sum(eigs_desc), 1e-12))\n\n\ndef ledger_hash(doc_dir: str | Path) -> str:\n    \"\"\"sha256(model_card.md || assumption_ledger.md), first 12 hex chars.\"\"\"\n    d = Path(doc_dir)\n    h = hashlib.sha256()\n    for name in (\"model_card.md\", \"assumption_ledger.md\"):\n        h.update((d / name).read_bytes())\n    return h.hexdigest()[:12]\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'de_formulas.py').write_text(harness_de_formulas)
assert hashlib.sha256((p/'de_formulas.py').read_bytes()).hexdigest()[:16] == '5dffb441b6382d00', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('de_formulas.py ok')

### harness: `simulator.py` (sha256 ef31ca2a201b7dc1)

In [ ]:
harness_simulator = "\"\"\"SCF Phase 1 simulator: DGP generation, estimators, spectral statistics.\n\nSchema: one parquet row per config x rep x estimator tag (research plan\nSection 10.3). Estimator tags used by the pilot:\n  ols, ridge_fixed (one row per lambda), pca_oracle (k=r), pca_onatski,\n  and, for cells with twin_gamma0=True, the gamma=0 twins ols_g0/ridge_fixed_g0\n  fitted on the SAME (Q, beta, f, u, eps) draw.\n\nAll randomness flows from np.random.SeedSequence([GLOBAL_SEED, cfg_hash, rep])\nso runs are reproducible and cells are resumable. Single-thread BLAS must be\nset by the launcher before numpy import (research plan Section 10.1).\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport math\nimport time\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom de_formulas import (\n    bbp_location,\n    bgn_overlap,\n    onatski_select,\n    tw_mu_sigma,\n    tw_threshold,\n)\n\nGLOBAL_SEED = 20260823\n\n\n@dataclass(frozen=True)\nclass Config:\n    n: int\n    p: int\n    r: int\n    l: tuple[float, ...]\n    theta: float\n    g: float = 1.0\n    sigma_u: float = 1.0\n    sigma_eps: float = 1.0\n    beta_kind: str = \"dense\"     # \"dense\" (A4a) | \"aligned\" (rung 4)\n    conf_kind: str = \"dense\"     # \"dense\" | \"sparse\" (rung 4 sparse loadings)\n    loading_kind: str = \"gauss\"  # \"gauss\" | \"rademacher_half\" (WP 2.4 V2)\n    error_law: str = \"gaussian\"  # \"gaussian\" | \"t5\" (WP 2.4 robustness)\n    hetero_u: bool = False       # row-heteroskedastic u (WP 2.4)\n    corr_factors: bool = False   # correlated f (WP 2.4; breaks Var(f)=I)\n    r_misspec: int = 0           # r perturbation consumed by priors (V5)\n    m2_treatment: bool = False   # generate D block (WP 2.4 / Phase 3 prep)\n    m2_tau: float = 1.0          # true scalar treatment coefficient\n    delta_g: float = 0.3         # ||delta|| for M2 weak-treatment alignment\n    twin_gamma0: bool = False    # run gamma=0 arm on common seeds\n    q_fixed: bool = False        # draw loading directions once per config\n    reps: int = 200\n    profile: str = \"\"\n    label: str = \"\"\n\n    @property\n    def c(self) -> float:\n        return self.p / self.n\n\n    @property\n    def cid(self) -> str:\n        payload = repr(\n            (self.n, self.p, self.r, self.l, round(self.theta, 6), self.g,\n             self.sigma_u, self.sigma_eps, self.beta_kind, self.conf_kind,\n             self.loading_kind, self.error_law, self.hetero_u,\n             self.corr_factors, self.r_misspec,\n             self.m2_treatment, round(self.m2_tau, 6), round(self.delta_g, 6),\n             self.profile, self.label)\n        )\n        return hashlib.sha256(payload.encode()).hexdigest()[:12]\n\n\ndef gamma_vector(cfg: Config) -> np.ndarray:\n    \"\"\"gamma = g * dir(theta) in factor coordinates (model card Section 2).\"\"\"\n    gam = np.zeros(cfg.r)\n    gam[0] = np.cos(cfg.theta)\n    if cfg.r > 1:\n        gam[1] = np.sin(cfg.theta)\n    return cfg.g * gam\n\n\ndef _rng(cfg: Config, rep: int) -> np.random.Generator:\n    return np.random.default_rng(\n        np.random.SeedSequence(entropy=[GLOBAL_SEED, int(cfg.cid[:8], 16), rep])\n    )\n\n\ndef gen_data(cfg: Config, rep: int):\n    \"\"\"Draw Q, beta, f, u, eps; return dict with X, Y and ingredients.\n\n    Phase 2 extensions (preregistration): sparse-confounding loadings\n    (conf_kind=\"sparse\": each loading column supported on ceil(sqrt(p))\n    coordinates, rescaled so Lambda'Lambda = diag(sigma_u^2 l_j) exactly),\n    aligned beta (beta_kind=\"aligned\": cosine 0.9 with the top factor\n    direction), t5 errors, row-heteroskedastic u, correlated factors, and an\n    M2 treatment block D = pi'X + delta'f + nu.\n    \"\"\"\n    rng = _rng(cfg, rep)\n    p, n, r = cfg.p, cfg.n, cfg.r\n    if cfg.q_fixed:\n        qrng = np.random.default_rng(\n            np.random.SeedSequence(entropy=[GLOBAL_SEED, int(cfg.cid[:8], 16), 2 ** 32])\n        )\n        A = qrng.standard_normal((p, r))\n    else:\n        A = rng.standard_normal((p, r))\n    Q, _ = np.linalg.qr(A)\n    svals = cfg.sigma_u * np.sqrt(np.asarray(cfg.l))\n    if cfg.conf_kind == \"sparse\":\n        # DISJOINT supports (r * ceil(sqrt(p)) <= p in all grids) so that\n        # Lambda'Lambda = diag(sigma_u^2 l_j) holds exactly and Ubasis stays\n        # orthonormal; rescaled columns preserve the spike strengths.\n        ksup = max(int(round(math.sqrt(p))), r)\n        Ldraw = qrng if cfg.q_fixed else rng\n        perm = Ldraw.permutation(p)\n        cols = []\n        for j in range(r):\n            support = perm[j * ksup:(j + 1) * ksup]\n            col = np.zeros(p)\n            col[support] = Ldraw.standard_normal(ksup)\n            cols.append(col)\n        Acol = np.column_stack(cols)\n        norms = np.linalg.norm(Acol, axis=0)\n        Lam = Acol * (svals / np.maximum(norms, 1e-12))[None, :]\n    elif cfg.loading_kind == \"rademacher_half\":\n        # WP 2.4 V2: Bernoulli-Rademacher loadings on a half support,\n        # rescaled so Lambda'Lambda = diag(sigma_u^2 l_j) exactly.\n        Ldraw = qrng if cfg.q_fixed else rng\n        signs = Ldraw.choice([-1.0, 1.0], size=(p, r))\n        support_mask = Ldraw.random((p, r)) < 0.5\n        Acol = np.where(support_mask, signs, 0.0)\n        norms = np.linalg.norm(Acol, axis=0)\n        norms = np.maximum(norms, 0.05 * math.sqrt(p / 2))  # avoid degenerate cols\n        norms = np.maximum(norms, 1e-6)\n        Lam = Acol * (svals / norms)[None, :]\n    else:\n        Lam = Q * svals  # p x r with scaled columns\n\n    if cfg.beta_kind == \"aligned\":\n        w = rng.standard_normal(p)\n        w -= Q @ (Q.T @ w)\n        nw = np.linalg.norm(w)\n        w = w / nw if nw > 1e-12 else w\n        beta = 0.9 * Q[:, 0] + math.sqrt(1.0 - 0.81) * w\n    else:\n        beta = rng.standard_normal(p)\n        beta /= np.linalg.norm(beta)\n\n    if cfg.corr_factors:\n        OmV, _ = np.linalg.qr(rng.standard_normal((r, r)))\n        om_evals = rng.uniform(0.5, 1.5, size=r)\n        Omega = (OmV * om_evals) @ OmV.T\n        f = rng.multivariate_normal(np.zeros(r), Omega, size=n)\n    else:\n        f = rng.standard_normal((n, r))\n\n    U_raw = rng.standard_normal((n, p))\n    if cfg.error_law == \"t5\":\n        # unit-variance t: t_5 / sqrt(5/3)\n        U_raw = rng.standard_t(5, size=(n, p)) / math.sqrt(5.0 / 3.0)\n        eps_core = rng.standard_t(5, size=n) / math.sqrt(5.0 / 3.0)\n    else:\n        eps_core = rng.standard_normal(n)\n    if cfg.hetero_u:\n        row_scale = np.sqrt((1.0 + rng.chisquare(1, size=(n, 1))) / 2.0)\n        U_raw = U_raw * row_scale\n    U = cfg.sigma_u * U_raw\n    X = f @ Lam.T + U\n    gam = gamma_vector(cfg)\n    noise_eps = cfg.sigma_eps * eps_core\n    out = dict(X=X, beta=beta, Q=Q, Lam=Lam, gam=gam, eps=noise_eps)\n    if cfg.m2_treatment:\n        pi = np.zeros(p)\n        kpi = max(3, p // 100)\n        pi[:kpi] = 1.0 / math.sqrt(kpi)\n        delta = rng.standard_normal(r)\n        delta *= cfg.delta_g / max(float(np.linalg.norm(delta)), 1e-12)\n        nu_ = cfg.sigma_eps * rng.standard_normal(n)\n        D = X @ pi + f @ delta + nu_\n        out.update(D=D, pi=pi, delta=delta)\n        out[\"Y\"] = cfg.m2_tau * D + X @ beta + f @ gam + noise_eps\n    else:\n        out[\"Y\"] = X @ beta + f @ gam + noise_eps\n    return out\n\n\ndef spectrum(X: np.ndarray):\n    \"\"\"Descending eigenpairs of the sample covariance on the R^p side.\n\n    For p > n the n-side Gram is diagonalized and eigenvectors are mapped back\n    via V_j = X' q_j / sqrt(n d_j) (exact normalization).\n    \"\"\"\n    n, p = X.shape\n    if p <= n:\n        d, W = np.linalg.eigh(X.T @ X / n)\n    else:\n        d, W = np.linalg.eigh(X @ X.T / n)\n        d = np.maximum(d, 1e-12)\n        W = X.T @ W / np.sqrt(n * d)\n    idx = np.argsort(d)[::-1]\n    return d[idx], W[:, idx]\n\n\ndef fit_ols(X, Y, eig) -> np.ndarray:\n    \"\"\"OLS (min-norm when p > n): V diag(1/d) V' X'Y/n or its wide analogue.\"\"\"\n    d, V = eig\n    n = X.shape[0]\n    return V @ ((V.T @ (X.T @ Y / n)) / d)\n\n\ndef fit_ridge(X, Y, eig, lam: float) -> np.ndarray:\n    \"\"\"(Sigmahat + lam I)^{-1} X'Y/n; exact for any p via the shared spectrum.\"\"\"\n    d, V = eig\n    n = X.shape[0]\n    rhs = V.T @ (X.T @ Y / n)\n    return V @ (rhs / (d + lam))\n\n\ndef fit_pca(X, Y, eig, k: int) -> np.ndarray:\n    \"\"\"Regress Y on the top-k sample PCs (k >= 1); k <= 0 gives zero vector.\"\"\"\n    if k <= 0:\n        return np.zeros(X.shape[1])\n    Vk = eig[1][:, :k]\n    delta, *_ = np.linalg.lstsq(X @ Vk, Y, rcond=None)\n    return Vk @ delta\n\n\ndef run_rep(cfg: Config, rep: int, lam_grid: tuple[float, ...]):\n    \"\"\"Run one replication; returns (rows, mean_diffs).\n\n    rows: list of dicts (one per estimator tag) with scalar metrics and shared\n    spectral statistics.\n    mean_diffs: tag -> (beta_hat - beta) vector, accumulated across reps by the\n    runner to estimate E[beta_hat] - beta without storing every draw.\n    \"\"\"\n    t0 = time.perf_counter()\n    data = gen_data(cfg, rep)\n    X, Y, beta, Q = data[\"X\"], data[\"Y\"], data[\"beta\"], data[\"Q\"]\n    n, p = X.shape\n    eig = spectrum(X)\n    d, V = eig\n    mu_np, sig_np = tw_mu_sigma(n, p)\n    lam_max = float(d[0])\n\n    stats_common = {\n        \"lam_max_cov\": lam_max,\n        \"tw_stat\": float((lam_max * n - mu_np) / sig_np),\n        \"outlier99\": bool(lam_max > tw_threshold(n, p, cfg.sigma_u)),\n        \"bbp_pred\": bbp_location(max(cfg.l), cfg.c, cfg.sigma_u),\n        \"xi1_pred\": bgn_overlap(max(cfg.l), cfg.c),\n    }\n    overlaps = []\n    for j in range(min(cfg.r, V.shape[1])):\n        overlaps.append(float((V[:, j] @ Q[:, j]) ** 2))\n\n    arms = [(\"ols\", np.nan)]\n    for lam in lam_grid:\n        arms.append((f\"ridge_fixed|{lam}\", lam))\n\n    def make_rows(suffix: str, Y_arm: np.ndarray):\n        out = []\n        diffs = {}\n        bh_ols = fit_ols(X, Y_arm, eig)\n        ests = [(f\"ols{suffix}\", np.nan, bh_ols)]\n        for _, lam in arms[1:]:\n            ests.append((f\"ridge_fixed{suffix}\", lam, fit_ridge(X, Y_arm, eig, lam)))\n        k_on = max(onatski_select(d), 0)\n        ests.append((f\"pca_onatski{suffix}\", np.nan, fit_pca(X, Y_arm, eig, k_on)))\n        for tag, lam, bh in ests:\n            diff = bh - beta\n            row = {\n                \"config_id\": cfg.cid,\n                \"rep\": rep,\n                \"seed\": GLOBAL_SEED,\n                \"estimator\": tag,\n                \"lambda\": lam,\n                \"k_select\": k_on if tag.startswith(\"pca\") else -1,\n                \"rel_err\": float(np.linalg.norm(diff)),\n                \"runtime_s\": 0.0,\n            }\n            row.update(stats_common)\n            for j in range(cfg.r):\n                ov = overlaps[j] if j < len(overlaps) else np.nan\n                row[f\"overlap{j + 1}\"] = ov\n            out.append(row)\n            key = tag if np.isnan(lam) else f\"{tag}@{lam}\"\n            diffs[key] = diff.astype(np.float64)\n        return out, diffs\n\n    rows, mean_diffs = make_rows(\"\", Y)\n\n    if cfg.twin_gamma0:\n        Y0 = X @ beta + data[\"eps\"]  # same seeds: identical everything but gamma link\n        rows0, diffs0 = make_rows(\"_g0\", Y0)\n        rows.extend(rows0)\n        mean_diffs.update(diffs0)\n\n    runtime = time.perf_counter() - t0\n    for r_ in rows:\n        r_[\"runtime_s\"] = runtime / len(rows)\n    return rows, mean_diffs\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'simulator.py').write_text(harness_simulator)
assert hashlib.sha256((p/'simulator.py').read_bytes()).hexdigest()[:16] == 'ef31ca2a201b7dc1', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('simulator.py ok')

### harness: `detection.py` (sha256 06586fe60b9f6573)

In [ ]:
harness_detection = "\"\"\"SCF Phase 2 detection statistics (WP 2.3). Frozen spec:\ndocs/detection_statistics.md. All statistics consume the shared spectrum\n(descending eigenpairs of X_c'X_c/n) plus the standardized response.\n\nPer-rep outputs are plain floats collected into parquet rows by the runner;\nthresholds follow the frozen spec (analytic variants computed here,\nMC-calibrated variants computed at analysis time from pooled null reps).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom de_formulas import (\n    TW1_Q95,\n    TW1_Q99,\n    maxz_null_var as df_maxz_null_var,\n    onatski_select,\n    tw_threshold,\n    tw_width_cov_scale,\n)\n\n\ndef ktop_default(eig, cap: int = 10) -> int:\n    \"\"\"Number of spike coordinates used by S2/B1: Onatski selection, min 1.\"\"\"\n    return int(min(max(onatski_select(eig[0]), 1), cap))\n\n\ndef compute_stats(\n    Xc: np.ndarray,\n    Yraw: np.ndarray,\n    eig,\n    sigma_u: float = 1.0,\n    ktop: int | None = None,\n):\n    \"\"\"Yraw is the centered raw response; it is standardized internally\n    (scale-invariant statistics).\"\"\"\n    sd_y = float(np.std(Yraw)) or 1.0\n    Ys = Yraw / sd_y\n    \"\"\"Return dict of detection statistics for one dataset.\n\n    Keys:\n      lam_max_cov, tw_stat, outlier99          -> S0 scree baseline\n      t_aug, s1_thresh_analytic                -> S1 aug_bbp\n      t_maxz, s2_thresh_bonf                   -> S2 maxz_cal\n      lss_rank1                                -> S3 reduction (rank-one LSS)\n      f_pcs                                    -> B1 partial F on PC scores\n      b_norm2, z_coords                        -> probe features / diagnostics\n      ktop                                     -> coordinate count used\n    \"\"\"\n    n, p = Xc.shape\n    c = p / n\n    d, V = eig\n    b = Xc.T @ Ys / n\n    if ktop is None:\n        ktop = ktop_default(eig)\n\n    # ---- S0: scree / TW99 visibility -------------------------------------\n    mu_np = None\n    lam_max = float(d[0])\n    from de_formulas import tw_mu_sigma\n\n    mu_np, sig_np = tw_mu_sigma(n, p)\n    tw_stat = float((lam_max * n - mu_np) / sig_np)\n    outlier99 = bool(lam_max > tw_threshold(n, p, sigma_u))\n\n    # ---- S2: calibrated max-z over spike coordinates ----------------------\n    # F12 (Erratum 1): Var(z_j) = (sigma_eps2 + d_j/c)/sigma_y2 with\n    # sigma_y2 = tr(Sigmahat)/p + sigma_eps2; scales from shared helper.\n    kk = min(ktop, len(d))\n    w = V[:, :kk].T @ b\n    z = np.sqrt(n) * w / np.sqrt(d[:kk])\n    from de_formulas import estimate_noise_scales\n\n    se2_hat, sigma_y2 = estimate_noise_scales(d, c)\n    var_cal = np.array([\n        df_maxz_null_var(dj, c, se2_hat, sigma_y2) for dj in d[:kk]\n    ])\n    t_maxz = float(np.max(np.abs(z) / np.sqrt(var_cal)))\n    from scipy.stats import norm\n\n    s2_thresh = float(norm.ppf(1.0 - 0.05 / (2 * kk)))\n\n    # ---- S1: augmented secular root ---------------------------------------\n    w_all = V.T @ b\n    dpos = np.maximum(d, 1e-300)\n    lo = float(d[0]) * (1.0 + 1e-9) + 1e-15\n    s_total = float(np.sum(w_all ** 2))\n    disc = (1.0 + lo) ** 2 - 4.0 * (lo + s_total)\n    hi = 0.5 * ((1.0 + lo) + np.sqrt(max(disc, 1e-300))) * (1.0 + 1e-6) + 1e-6\n\n    def f_aug(lam):\n        return lam - 1.0 - float(np.sum(w_all ** 2 / (lam - dpos)))\n\n    flo = f_aug(lo)\n    if flo > 0:  # degenerate: root pinned at top edge\n        t_aug = lo\n    else:\n        for _ in range(200):\n            mid = 0.5 * (lo + hi)\n            if f_aug(mid) > 0:\n                hi = mid\n            else:\n                lo = mid\n            if hi - lo < 1e-11 * max(1.0, hi):\n                break\n        t_aug = 0.5 * (lo + hi)\n\n    # analytic H0 threshold: plug-in null root + TW95 width at dims (n, p+1)\n    from de_formulas import aug_null_root\n\n    lam0 = aug_null_root(d, max(sigma_y2, 1e-6))\n    width = tw_width_cov_scale(n, p + 1, sigma_u) * max(sigma_y2, 1.0) ** 0.5\n    s1_thresh = float(lam0 + TW1_Q95 * width)\n\n    # ---- S3 reduction: rank-one LSS ----------------------------------------\n    lss_rank1 = float(n * np.sum(w_all ** 2 / dpos))\n\n    # ---- B1: partial F of Y on top-ktop PC scores ---------------------------\n    S = Xc @ V[:, :kk]\n    rss_r = float(np.sum(Ys ** 2))\n    coef, *_ = np.linalg.lstsq(S, Ys, rcond=None)\n    rss_f = float(np.sum((Ys - S @ coef) ** 2))\n    denom = rss_f / max(n - kk, 1)\n    f_pcs = ((rss_r - rss_f) / kk) / max(denom, 1e-300)\n\n    return {\n        \"lam_max_cov\": lam_max,\n        \"tw_stat\": tw_stat,\n        \"outlier99\": outlier99,\n        \"t_aug\": float(t_aug),\n        \"s1_thresh_analytic\": s1_thresh,\n        \"lam0_plugin\": float(lam0),\n        \"t_maxz\": t_maxz,\n        \"s2_thresh_bonf\": s2_thresh,\n        \"lss_rank1\": lss_rank1,\n        \"f_pcs\": float(f_pcs),\n        \"b_norm2\": float(np.sum(b ** 2)),\n        \"z_top\": float(z[0]),\n        \"ktop\": int(kk),\n    }\n\n\ndef rejections(stats: dict) -> dict:\n    \"\"\"Frozen decision rules applied to a stats dict.\"\"\"\n    return {\n        \"rej_s0_tw99\": bool(stats[\"outlier99\"]),\n        \"rej_s1_analytic\": bool(stats[\"t_aug\"] > stats[\"s1_thresh_analytic\"]),\n        \"rej_s2_bonf\": bool(stats[\"t_maxz\"] > stats[\"s2_thresh_bonf\"]),\n        \"rej_b1_f95\": bool(stats[\"f_pcs\"] > _f_ppf95(stats[\"ktop\"], 10_000)),\n    }\n\n\n_F95_CACHE: dict[tuple[int, int], float] = {}\n\n\ndef mc_thresholds(null_rows: list[dict], q: float = 0.95) -> dict[str, float]:\n    \"\"\"Monte-Carlo decision thresholds from pooled null reps (frozen rule).\n\n    The gate statistics are the MC-calibrated rejections; the analytic\n    variants are co-recorded but reported as-is (v1 analytic widths are\n    known-miscalibrated for the augmented deformation: measured sd/width\n    ratio ~ 8.5 at n=600 before any sweep data, see Erratum notes in\n    docs/detection_statistics.md).\n    \"\"\"\n    t_aug = np.array([r[\"t_aug\"] for r in null_rows])\n    t_z = np.array([r[\"t_maxz\"] for r in null_rows])\n    lss = np.array([r[\"lss_rank1\"] for r in null_rows])\n    f_pcs = np.array([r[\"f_pcs\"] for r in null_rows])\n    return {\n        \"q_t_aug\": float(np.quantile(t_aug, q)),\n        \"q_t_maxz\": float(np.quantile(t_z, q)),\n        \"q_lss\": float(np.quantile(lss, q)),\n        \"q_f\": float(np.quantile(f_pcs, q)),\n    }\n\n\ndef _f_ppf95(df1: int, df2: int) -> float:\n    key = (df1, min(df2, 100000))\n    if key not in _F95_CACHE:\n        from scipy.stats import f as f_dist\n\n        _F95_CACHE[key] = float(f_dist.ppf(0.95, df1, df2))\n    return _F95_CACHE[key]\n\n\n# ---------------------------------------------------------------------------\n# Numerical Le Cam probe helpers (frozen feature map; lazy sklearn import)\n# ---------------------------------------------------------------------------\n\n\ndef probe_features(eig, stats: dict, r_hint: int = 4) -> np.ndarray:\n    \"\"\"Frozen feature map (docs/detection_statistics.md): 12 + 4 values.\"\"\"\n    d = eig[0]\n    logs = np.log(np.maximum(d[:10], 1e-12))\n    if len(logs) < 10:\n        logs = np.pad(logs, (0, 10 - len(logs)))\n    extra = [\n        float(np.sum(d)),\n        stats[\"t_aug\"],\n        stats[\"b_norm2\"],\n        stats[\"z_top\"],\n    ]\n    feats = np.concatenate([logs, np.array(extra)])\n    if r_hint > 4:\n        feats = np.concatenate([feats, np.zeros(4)])\n    return feats\n\n\ndef lecam_auc(H0_feats: np.ndarray, H1_feats: np.ndarray, seed: int = 0):\n    \"\"\"GBM + median-heuristic MMD two-sample probe; returns (auc_gbm, auc_mmd).\n\n    Computational probe ONLY (not information-theoretic): declaration\n    threshold AUC <= 0.55 for both probes per frozen spec. 50/50 held-out\n    split, stratified; GBM = HistGradientBoostingClassifier defaults.\n    \"\"\"\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    from sklearn.metrics import roc_auc_score\n    from sklearn.model_selection import train_test_split\n\n    X = np.vstack([H0_feats, H1_feats])\n    y = np.concatenate([np.zeros(len(H0_feats)), np.ones(len(H1_feats))])\n    Xtr, Xte, ytr, yte = train_test_split(\n        X, y, test_size=0.5, random_state=seed, stratify=y\n    )\n    clf = HistGradientBoostingClassifier(random_state=seed)\n    clf.fit(Xtr, ytr)\n    auc_gbm = float(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]))\n\n    mu = Xtr.mean(axis=0)\n    sd = Xtr.std(axis=0) + 1e-12\n    A = (Xtr[ytr == 1] - mu) / sd\n    B = (Xtr[ytr == 0] - mu) / sd\n    Z = (Xte - mu) / sd\n\n    def kmat(A_, B_):\n        aa = np.sum(A_ * A_, axis=1)[:, None]\n        bb = np.sum(B_ * B_, axis=1)[None, :]\n        return np.maximum(aa + bb - 2.0 * (A_ @ B_.T), 0.0)\n\n    med = float(np.median(kmat(A[:500], B[:500]))) or 1.0\n    gamma = 1.0 / med\n    scores = np.exp(-gamma * kmat(Z, A)).mean(axis=1) - np.exp(\n        -gamma * kmat(Z, B)\n    ).mean(axis=1)\n    auc_mmd = float(roc_auc_score(yte, scores))\n    return auc_gbm, auc_mmd\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'detection.py').write_text(harness_detection)
assert hashlib.sha256((p/'detection.py').read_bytes()).hexdigest()[:16] == '06586fe60b9f6573', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('detection.py ok')

### harness: `estimators.py` (sha256 7e27f25b2330b8e1)

In [ ]:
harness_estimators = "\"\"\"SCF Phase 2 estimator suite (WP 2.2). Frozen roster:\ndocs/phase2_preregistration.md. All estimators consume the shared spectrum\n(descending eigenpairs of X_c'X_c/n) so per-rep cost is one eigendecomposition\nplus cheap per-estimator algebra.\n\nEvery estimator returns a coefficient vector in R^p. Tuning budgets are fixed\nmodule constants (logged into parquet rows by the runner) per the fair\ncomparison protocol (research plan Section 8.3).\n\nSources pinned for baselines:\n- Cevid, Buhlmann, Meinshausen (AoS 2020), arXiv:1811.05352: Trim transform\n  d~_i = min(d_i^svd, median(d^svd)), F = U diag(d~/d) U', regression on\n  transformed data (their eqs. 3.1-3.3).\n- Nava, Buhlmann, Sigrist (2026), arXiv:2607.09371: LAVA-type spectral loss\n  w_i = sigma_e2/(sigma_r2 s_i^2 + sigma_e2); variance components by marginal\n  likelihood ell(theta) (their Section 4.2.1); linear-base-learner boosting\n  path alpha_j(m) = (z_j/d_j)(1-(1-nu w_j)^m) (their Section 3 recursion);\n  stopping by BLUP-corrected K-fold CV (their Section 4.2.2).\n\"\"\"\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\n\nfrom de_formulas import (\n    bbp_invert,\n    lava_weights,\n    seb_predicted_mse,\n    sdboost_path_coefficients,\n    trim_weights,\n)\n\nRIDGE_LAM_GRID = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0)\nSDBOOST_NU = 0.1\nSDBOOST_M_GRID = tuple(int(m) for m in np.unique(np.round(np.logspace(0, 3.2, 14))))\nEB_TAU_FRAC_GRID = tuple(\n    float(f) for f in np.concatenate([np.linspace(0.05, 0.95, 19), [1.5, 2.0, 4.0]])\n)\nN_FOLDS = 5\n\n\n# ---------------------------------------------------------------------------\n# shared pieces\n# ---------------------------------------------------------------------------\n\n\ndef center_columns(X: np.ndarray) -> np.ndarray:\n    return X - X.mean(axis=0, keepdims=True)\n\n\ndef standardize_response(Y: np.ndarray) -> np.ndarray:\n    y = Y - Y.mean()\n    sd = float(np.std(Y)) or 1.0\n    return y / sd\n\n\ndef spectral_fit(d: np.ndarray, V: np.ndarray, Xty_over_n: np.ndarray, w: np.ndarray):\n    \"\"\"beta = V diag(w_j / d_j) V' (X'Y/n); w_j = weight on covariance scale.\"\"\"\n    coef = V.T @ Xty_over_n\n    return V @ ((w * coef) / d)\n\n\ndef kfold_indices(n: int, k: int, rng: np.random.Generator):\n    idx = rng.permutation(n)\n    return np.array_split(idx, k)\n\n\n# ---------------------------------------------------------------------------\n# roster members; signature (Xc, Ys, eig, rng) -> beta_hat\n# Xc column-centered design, Ys standardized response, eig = (d desc, V)\n# ---------------------------------------------------------------------------\n\n\ndef est_ols(Xc, Ys, eig, rng=None):\n    d, V = eig\n    return fit_min_norm(Xc, Ys, eig)\n\n\ndef fit_min_norm(Xc, Ys, eig):\n    d, V = eig\n    return spectral_fit(d, V, Xc.T @ Ys / len(Ys), np.ones_like(d))\n\n\ndef loo_scores_spectral(\n    U: np.ndarray, Ys: np.ndarray, d: np.ndarray, weight_fun, grid\n):\n    \"\"\"Exact leave-one-out CV scores for any diagonal spectral fit.\n\n    Any member of the family beta = V diag(w_j/d_j) U'Y induces the fitted\n    operator Yhat = U diag(w_j) U' Y, whose hat-matrix diagonal is\n    h_ii(w) = sum_j w_j u_ij^2. The exact LOO identity\n    r_i^(-i) = (y_i - yhat_i) / (1 - h_ii(w)) then gives\n    LOO-MSE(w) = mean(r_i^(-i)^2) at O(n k) per grid point, replacing\n    fold-wise eigendecompositions (documented equivalence; budgets logged).\n    Ridge corresponds to w_j = d_j/(d_j + lam) on the covariance scale.\n    \"\"\"\n    n = len(Ys)\n    z = U.T @ Ys\n    U2 = U ** 2\n    scores = []\n    for param in grid:\n        w = weight_fun(param)\n        yhat = U @ (w * z)\n        h = U2 @ w\n        denom = np.maximum(1.0 - h, 1e-12)\n        r_loo = (Ys - yhat) / denom\n        scores.append(float(np.mean(r_loo ** 2)))\n    return scores\n\n\ndef left_factors(Xc: np.ndarray, eig):\n    \"\"\"Left singular vectors U (n x k) from the shared spectrum.\"\"\"\n    d, V = eig\n    return Xc @ V / np.sqrt(len(Xc) * d)[None, :]\n\n\ndef _cv_ridge(Xc, Ys, eig, rng, lam_grid=RIDGE_LAM_GRID):\n    \"\"\"Ridge with lambda by EXACT LOO-CV over the frozen grid (see\n    loo_scores_spectral). Returns (beta_hat, info).\"\"\"\n    n = len(Ys)\n    d, V = eig\n    U = left_factors(Xc, eig)\n\n    def wf(lam):\n        return d / (d + lam)\n\n    scores = loo_scores_spectral(U, Ys, d, wf, lam_grid)\n    lam = float(lam_grid[int(np.argmin(scores))])\n    return _ridge_on_spectrum(d, V, Xc, Ys, lam), {\"lam\": lam, \"grid\": len(lam_grid)}\n\n\ndef _ridge_on_spectrum(d, V, Xc, Ys, lam):\n    n = len(Ys)\n    rhs = V.T @ (Xc.T @ Ys / n)\n    return V @ (rhs / (d + lam))\n\n\ndef est_ridge_cv(Xc, Ys, eig, rng):\n    return _cv_ridge(Xc, Ys, eig, rng)[0]\n\n\ndef est_pca_k(Xc, Ys, eig, rng=None, k: int = 0):\n    d, V = eig\n    w = np.zeros_like(d)\n    w[:k] = 1.0\n    return spectral_fit(d, V, Xc.T @ Ys / len(Ys), w)\n\n\ndef est_cevid_default(Xc, Ys, eig, rng=None):\n    \"\"\"Trim transform with tau = median singular value + OLS on transformed data.\n\n    On the covariance scale their singular-value cap at median(d_svd) is\n    w_j = min(1, median(d_svd)^2 / (n d_cov_j)); the global factor 1/n is\n    absorbed by the fit invariance, so w_j = min(1, median_svd^2/(n d_j)).\n    \"\"\"\n    n = len(Ys)\n    d, V = eig\n    med_svd2 = n * float(np.median(d))\n    w = trim_weights(d * n, med_svd2)\n    return spectral_fit(d, V, Xc.T @ Ys / n, w)\n\n\ndef est_lava_transform_ols(Xc, Ys, eig, rng=None, rho: float | None = None):\n    \"\"\"LAVA transform at rho = n*lam2 chosen by their lambda2 rule proxy.\n\n    Their suggested rule sets n lam2 near bulk scale (sqrt(n)+sqrt(p))^2;\n    we use exactly that as the default tuning (documented transcription of\n    their Section 3 discussion via Nava et al. eq. 12 context).\n    \"\"\"\n    n, p = Xc.shape\n    d, V = eig\n    if rho is None:\n        rho = float((math.sqrt(n) + math.sqrt(p)) ** 2)\n    w = lava_weights(d, rho)\n    return spectral_fit(d, V, Xc.T @ Ys / n, w)\n\n\ndef sdboost_marginal_ll(theta_log: np.ndarray, s: np.ndarray, z: np.ndarray,\n                        perp2: float, n: int) -> float:\n    sr2, se2 = np.exp(theta_log)\n    var = se2 + sr2 * s\n    ll = -0.5 * np.sum(np.log(var) + z ** 2 / var)\n    ll += -0.5 * (n - len(s)) * math.log(se2) - perp2 / (2 * se2)\n    return ll\n\n\ndef fit_sdboost_linear_eb(Xc, Ys, eig, rng, nu: float = SDBOOST_NU):\n    \"\"\"SDBoost linear special case: EB(LAVA weights) + BLUP-corrected CV stop.\n\n    Faithful composition of Nava et al.: (1) variance components by marginal\n    likelihood on the XX' spectrum (their 4.2.1, single-shot version: the\n    alternating theta/f updates are initialized at the ridge-small fit);\n    (2) boosting path coefficients (their Section 3 recursion, exact);\n    (3) stopping time by BLUP-corrected K-fold CV (their 4.2.2 formula).\n    Returns (beta_hat, info dict).\n    \"\"\"\n    from scipy.optimize import minimize\n\n    if rng is None:\n        rng = np.random.default_rng(0)\n    n, p = Xc.shape\n    d, V = eig\n    U = Xc @ V / np.sqrt(n * d)[None, :]\n    z = U.T @ Ys  # coordinates along left singular vectors (n-side)\n    # variance components: initialize around ridge-small residual split\n    lam_small = 0.01 * float(np.median(d))\n    resid = Ys - _ridge_on_spectrum(d, V, Xc, Ys, lam_small) @ Xc.T\n    perp2 = float(resid @ resid)\n    s_full = n * d\n    # coarse 2-D grid over log variance components, then local refine\n    grid_sr = np.linspace(-16.0, 6.0, 23)\n    grid_se = np.linspace(-16.0, 6.0, 23)\n    ll_vals = np.array([\n        [-sdboost_marginal_ll(np.array([a, b]), s_full, z, perp2, n)\n         for b in grid_se] for a in grid_sr\n    ])\n    i0 = np.unravel_index(np.argmin(ll_vals), ll_vals.shape)\n    x0 = np.array([grid_sr[i0[0]], grid_se[i0[1]]])\n    from scipy.optimize import minimize\n\n    res = minimize(\n        lambda t: -sdboost_marginal_ll(t, s_full, z, perp2, n),\n        x0, method=\"L-BFGS-B\", bounds=[(-30.0, 10.0), (-30.0, 10.0)],\n    )\n    sr2, se2 = np.exp(res.x)\n\n    def path_beta(m, d_svd, w, z_tr):\n        alpha = sdboost_path_coefficients(z_tr, d_svd, w, m, nu)\n        return V @ alpha\n\n    # BLUP-corrected CV over the iteration grid. Frozen approximation: fold\n    # random-effect operators use the FULL-data left singular subspace U\n    # (rows masked to the training fold) instead of refolding spectra; the\n    # kinship operator is dominated by the top-left-singular subspace, and\n    # this removes all fold-wise eigendecompositions (documented in the\n    # preregistration deviation log if it materially changes results).\n    n_folds = N_FOLDS\n    folds = kfold_indices(n, n_folds, rng)\n    # cache fold-invariant quantities: cross Grams and training coordinates\n    fold_cache = []\n    for f in folds:\n        mask = np.ones(n, bool)\n        mask[f] = False\n        Xtr = Xc[mask]\n        n_tr = int(mask.sum())\n        fold_cache.append({\n            \"mask\": mask, \"Xtr\": Xtr, \"n_tr\": n_tr,\n            \"z_tr\": U[mask].T @ Ys[mask],\n            \"cross\": Xc[f] @ Xtr.T,\n        })\n    cv_scores = []\n    for m in SDBOOST_M_GRID:\n        tot = 0.0\n        for fc in fold_cache:\n            mask, Xtr, n_tr = fc[\"mask\"], fc[\"Xtr\"], fc[\"n_tr\"]\n            Ytr = Ys[mask]\n            w_tr = 1.0 / (1.0 + (sr2 / se2) * (n_tr * d))\n            bm = path_beta(m, np.sqrt(n_tr * d), w_tr, fc[\"z_tr\"])\n            rtr = Ytr - Xtr @ bm\n            # Woodbury on Sig_tr = se2 I + sr2 * Utr S_tr Utr', shared-U approx\n            s_tr = n_tr * d\n            coef_u = U[mask].T @ rtr\n            shrink = (sr2 * s_tr) / (se2 + sr2 * s_tr)\n            sol = (rtr - U[mask] @ (shrink * coef_u)) / se2\n            blup = sr2 * (fc[\"cross\"] @ sol)\n            pred = Xc[~mask] @ bm + blup\n            tot += float(np.mean((Ys[~mask] - pred) ** 2))\n        cv_scores.append(tot / n_folds)\n    m_star = SDBOOST_M_GRID[int(np.argmin(cv_scores))]\n    w_full = 1.0 / (1.0 + (sr2 / se2) * (n * d))\n    beta = path_beta(m_star, np.sqrt(n * d), w_full, z)\n    info = {\"sr2\": sr2, \"se2\": se2, \"m\": m_star, \"grid\": len(SDBOOST_M_GRID)}\n    return beta, info\n\n\ndef spectrum_of(Xc: np.ndarray):\n    \"\"\"Descending eigenpairs of X_c'X_c/n reusing simulator.spectrum.\"\"\"\n    from simulator import spectrum\n\n    return spectrum(Xc)\n\n\n# ---------------------------------------------------------------------------\n# OUR estimator: SEB-tuned soft trim\n# ---------------------------------------------------------------------------\n\n\ndef estimate_spike_profile(d: np.ndarray, c: float, kmax: int = 10):\n    \"\"\"Map sample eigenvalues to population spike estimates l_hat_j.\n\n    Walk down the top kmax eigenvalues; while bbp_invert yields l > sqrt(c),\n    record it; stop at the first non-outlier. Subcritical directions get\n    l_hat = small floor (1e-3) so capture ~ 1/c-ish behavior is preserved.\n    \"\"\"\n    l_hats = []\n    for dj in d[:kmax]:\n        lj = bbp_invert(dj, c)\n        if np.isnan(lj):\n            break\n        l_hats.append(lj)\n    return np.array(l_hats if l_hats else [1e-3])\n\n\ndef estimate_gamma2(\n    Xc: np.ndarray, Ys: np.ndarray, eig, l_hat: np.ndarray, c: float\n) -> np.ndarray:\n    \"\"\"Cross-moment mixture estimate of gamma_j^2 (SEB plug-in).\n\n    z_j = sqrt(n) v_j'b / sqrt(d_j) has null variance\n    var_cal = (se2 + d_j/c)/sigma_y2 (F12 Erratum 1) and H1 mean\n    mu_j = sqrt(n l_j) g dir_j / (sqrt(1+l_j) sigma_y). Positive-part moment\n    inversion:\n\n        ghat_j^2 = max(0, z_j^2 - var_cal) (1 + l_hat_j) sigma_y^2 / (n l_hat_j),\n\n    conservative since dir^2 <= 1 and chi-square fluctuations inflate it.\n    Frozen as the SEB estimator; scales from estimate_noise_scales so\n    detection and tuning agree.\n    \"\"\"\n    n = len(Ys)\n    d, V = eig\n    from de_formulas import estimate_noise_scales\n\n    se2, sigma_y2 = estimate_noise_scales(d, c)\n    sd_y = float(np.std(Ys)) or 1.0\n    b = Xc.T @ Ys / (n * sd_y)\n    out = []\n    for j, lj in enumerate(l_hat):\n        wj = float(V[:, j] @ b)\n        zj2 = n * wj ** 2 / d[j]\n        var_cal = (se2 + d[j] / c) / sigma_y2 ** 2\n        excess = max(0.0, zj2 - var_cal)\n        denom = n * max(lj, 1e-6)\n        out.append(excess * (1.0 + lj) * sigma_y2 / denom)\n    return np.array(out)\n\n\ndef _noise_scale_for(eig, c: float) -> float:\n    \"\"\"Shared bulk-median noise-scale estimate (detection/tuner agreement).\"\"\"\n    from de_formulas import estimate_noise_scales\n\n    return estimate_noise_scales(eig[0], c)[0]\n\n\ndef est_eb_spectral(Xc, Ys, eig, rng, tau_override: float | None = None,\n                    kmax_hint: int | None = None):\n    \"\"\"SEB soft-trim estimator (ours). Returns (beta_hat, info).\n\n    kmax_hint bounds how many sample eigenvalues may be read as supercritical\n    spikes (V5 r-misspecification cells consume cfg.r + cfg.r_misspec).\n    \"\"\"\n    n, p = Xc.shape\n    c = p / n\n    d, V = eig\n    l_hat = estimate_spike_profile(d, c, kmax=kmax_hint or 10)\n    g2_hat = estimate_gamma2(Xc, Ys, eig, l_hat, c)\n    se2 = _noise_scale_for(eig, c)\n    if tau_override is not None:\n        tau = tau_override\n    else:\n        taus = [f * float(np.median(d)) for f in EB_TAU_FRAC_GRID]\n        vals = [\n            seb_predicted_mse(l_hat, g2_hat, d, t, c, n, se2) for t in taus\n        ]\n        tau = taus[int(np.argmin(vals))]\n    w = trim_weights(d, tau)\n    beta = spectral_fit(d, V, Xc.T @ Ys / n, w)\n    info = {\n        \"tau\": tau,\n        \"l_hat\": l_hat.tolist(),\n        \"g2_hat\": g2_hat.tolist(),\n        \"grid\": len(EB_TAU_FRAC_GRID),\n    }\n    return beta, info\n\n\ndef est_eb_cv_tau(Xc, Ys, eig, rng):\n    \"\"\"Ablation no-EB: same soft-trim family, tau by EXACT LOO prediction-CV\n    (loo_scores_spectral with w = min(1, tau/d)). Isolates what the SEB\n    causal objective adds over plain predictive tuning.\"\"\"\n    n, p = Xc.shape\n    d, V = eig\n    U = left_factors(Xc, eig)\n    taus = [fr * float(np.median(d)) for fr in EB_TAU_FRAC_GRID]\n    scores = loo_scores_spectral(\n        U, Ys, d, lambda t: trim_weights(d, t), taus\n    )\n    tau = float(taus[int(np.argmin(scores))])\n    w = trim_weights(d, tau)\n    return spectral_fit(d, V, Xc.T @ Ys / n, w), {\"tau\": tau}\n\n\n# ---------------------------------------------------------------------------\n# oracle variants (diagnostic upper bounds only; plan Section 8.3)\n# ---------------------------------------------------------------------------\n\n\ndef make_oracle_estimator(Q: np.ndarray, gam: np.ndarray):\n    \"\"\"oracle_gamma: beta_hat = beta_ols - (Q Q' beta_ols + Q gamma), i.e.,\n    OLS minus the true loading-subspace bias contribution. Upper bound for\n    what any spectral method could achieve on the confounding component.\n    \"\"\"\n\n    def f(Xc, Ys, eig, rng=None):\n        b_ols = fit_min_norm(Xc, Ys, eig)\n        return b_ols - Q @ (Q.T @ b_ols) - Q @ gam\n\n    return f\n\n\ndef eb_oracle_tau_factory(l_true: np.ndarray, gam_true: np.ndarray):\n    \"\"\"eb_oracle_tau: same weight family and objective as est_eb_spectral,\n    but fed the TRUE (l, gamma^2) instead of plug-in estimates.\n\n    Frozen definition (preregistration): the oracle ablation isolates tuner\n    ESTIMATION error, not objective mismatch. The objective is\n    seb_predicted_mse with true parameters; variance/noise scale still uses\n    the observable bulk estimate (_noise_scale_for) so only the confounding\n    geometry is oracular.\n    \"\"\"\n\n    def f(Xc, Ys, eig, rng=None):\n        n, p = Xc.shape\n        c = p / n\n        d, V = eig\n        se2 = _noise_scale_for(eig, c)\n        g2_true = np.asarray(gam_true, float) ** 2\n        taus = [fr * float(np.median(d)) for fr in EB_TAU_FRAC_GRID]\n        vals = [seb_predicted_mse(l_true, g2_true, d, t, c, n, se2) for t in taus]\n        tau = taus[int(np.argmin(vals))]\n        w = trim_weights(d, tau)\n        return spectral_fit(d, V, Xc.T @ Ys / n, w)\n\n    return f\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'estimators.py').write_text(harness_estimators)
assert hashlib.sha256((p/'estimators.py').read_bytes()).hexdigest()[:16] == '7e27f25b2330b8e1', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('estimators.py ok')

### harness: `benchmarks_data.py` (sha256 a4bd80f13fe03ca6)

In [ ]:
harness_benchmarks_data = "\"\"\"SCF Phase 3 WP 3.1/WP 3.2: benchmark dataset acquisition, frozen\npreprocessing, and spectral audit (read-only wrt raw sources).\n\nFamilies (research plan Section 7, Phase 3 amendment):\n  A addneuromed : GEO GSE63060 + GSE63061 blood gene expression (Illumina\n                  HumanHT-12 v3 / v4), the two series ARE the processing\n                  batches of the AddNeuroMed cohort. Provider NCBI GEO.\n  B ihdp        : IHDP covariate benchmark (Hill 2011 npci covariates),\n                  featurized to high dimension with seeded random Fourier\n                  features. Public benchmark mirror.\n  C k401k       : wooldridge::k401ksubs household cross-section (Rdatasets\n                  CC0 mirror); M2 treatment block (e401k -> nettfa).\n\nFrozen preprocessing (recorded in configs/benchmarks_frozen.yaml BEFORE any\ncomparative result):\n  A: intersect ILMN probe IDs across the two series, average duplicate probe\n     IDs within each series, drop probes with missing values, apply\n     log2(x+1) iff the global max exceeds 25, z-score each probe on the\n     POOLED sample (batch mean shifts are preserved by design), keep the top\n     P probes by pooled variance.\n     A_main: P = 2000, all samples            (n = 717, c = 2.79)\n     A_sub : P = 1800, batch-2 samples only   (n = 388, c = 4.64)\n  B/C: standardize continuous covariates (binary kept as 0/1), then a seeded\n     random Fourier map Z = sqrt(2/P) cos(X W + b), W ~ N(0, h^-2 I_d) with\n     h the median-heuristic bandwidth on a seeded subsample.\n     B_main: n = 747, P = 750  (c = 1.00)   B_wide: P = 150 (c = 0.20)\n     C_main: n = 800 seeded subsample, P = 1600 (c = 2.00)\n     C_wide: same rows, P = 160 (c = 0.20)\n\nSpectral audit per config (consumed later by the frontier machinery):\n  se2_hat (shared bulk-median estimator), unit-scaled spectrum, Onatski\n  r_hat, BBP-inverted spike estimates l_hat_j, TW statistic of lambda_max vs\n  the white-noise threshold.\n\nOutputs:\n  data/benchmarks/<config>.npz      processed designs (+ labels/meta)\n  data/benchmarks/spectral_audit.json\n\nThis script never writes into data/benchmarks/raw/.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nRAW = ROOT / \"data\" / \"benchmarks\" / \"raw\"\nOUT = ROOT / \"data\" / \"benchmarks\"\nGLOBAL_SEED = 20260823\n\n\n# ---------------------------------------------------------------------------\n# Family A: AddNeuroMed series matrices\n# ---------------------------------------------------------------------------\n\n\ndef parse_geo_series(path: Path):\n    \"\"\"Return (probes, values [n x p], meta list[dict]) from one matrix.\"\"\"\n    samples, chars_rows, expr_rows = [], [], []\n    in_table = False\n    header = None\n    with gzip.open(path, \"rt\") as fh:\n        for line in fh:\n            line = line.rstrip(\"\\n\")\n            if line.startswith(\"!Sample_geo_accession\"):\n                samples = [c.strip('\"') for c in line.split(\"\\t\")[1:]]\n            elif line.startswith(\"!Sample_characteristics_ch1\"):\n                chars_rows.append(line.split(\"\\t\")[1:])\n            elif line.startswith(\"!series_matrix_table_begin\"):\n                in_table = True\n                header = next(fh).rstrip(\"\\n\").split(\"\\t\")\n            elif in_table:\n                if line.startswith(\"!series_matrix_table_end\"):\n                    break\n                expr_rows.append(line.split(\"\\t\"))\n    # metadata: characteristic cells may hold one field per cell; concatenate\n    # everything observed for sample i and split on ':' prefixes.\n    flat = [\"\"] * len(samples)\n    for row in chars_rows:\n        if len(row) == len(samples):\n            for i, v in enumerate(row):\n                flat[i] += v.strip('\"').strip() + \"; \"\n    meta = []\n    for blob in flat:\n        d = {}\n        for part in blob.split(\";\"):\n            if \":\" in part:\n                k, _, v = part.partition(\":\")\n                d[k.strip().lower()] = v.strip()\n        meta.append({\"status\": d.get(\"status\"), \"age\": d.get(\"age\"),\n                     \"gender\": d.get(\"gender\")})\n\n    # GEO layout: rows are probes (ID_REF), columns are samples.\n    probes = [r[0].strip('\"') for r in expr_rows]\n    vals_t = np.empty((len(expr_rows), len(samples)), dtype=np.float32)\n    for i, parts in enumerate(expr_rows):\n        vals_t[i] = np.asarray(parts[1:], dtype=np.float32)\n    return probes, vals_t.T, meta\n\n\ndef dedup_columns(probes, V):\n    \"\"\"Average duplicated probe IDs within one series.\"\"\"\n    uniq, inv = np.unique(np.asarray(probes), return_inverse=True)\n    if len(uniq) == len(probes):\n        return probes, V\n    out = np.zeros((V.shape[0], len(uniq)), dtype=V.dtype)\n    cnt = np.bincount(inv)\n    for j in range(V.shape[1]):\n        out[:, inv[j]] += V[:, j]\n    return [str(u) for u in uniq], out / cnt[None, :]\n\n\ndef build_addneuromed():\n    p60, v60, m60 = parse_geo_series(RAW / \"GSE63060_series_matrix.txt.gz\")\n    p61, v61, m61 = parse_geo_series(RAW / \"GSE63061_series_matrix.txt.gz\")\n    p60, v60 = dedup_columns(p60, v60)\n    p61, v61 = dedup_columns(p61, v61)\n    shared = sorted(set(p60) & set(p61))\n    idx60 = {p: i for i, p in enumerate(p60)}\n    idx61 = {p: i for i, p in enumerate(p61)}\n    cols60 = np.array([idx60[p] for p in shared])\n    cols61 = np.array([idx61[p] for p in shared])\n    A = np.vstack([v60[:, cols60], v61[:, cols61]])\n    batch = np.concatenate([np.zeros(v60.shape[0], int),\n                            np.ones(v61.shape[0], int)])\n    meta_all = m60 + m61\n    status = np.array([m[\"status\"] for m in meta_all])\n    age = np.array([float(m[\"age\"]) if m[\"age\"] else np.nan\n                    for m in meta_all])\n    gender = np.array([m[\"gender\"] for m in meta_all])\n\n    bad_col = np.isnan(A).any(axis=0)\n    bad_row = np.isnan(A).any(axis=1)\n    keep_row = ~bad_row & ~np.isnan(age) & (status != None)  # noqa: E711\n    A = A[keep_row][:, ~bad_col]\n    batch, status, age = batch[keep_row], status[keep_row], age[keep_row]\n\n    if float(A.max()) > 25.0:\n        A = np.log2(A + 1.0)\n    mu, sd = A.mean(axis=0, keepdims=True), A.std(axis=0, keepdims=True)\n    Z = (A - mu) / np.maximum(sd, 1e-12)\n    var_order = np.argsort(-Z.var(axis=0), kind=\"stable\")\n\n    out = {}\n    for name, P, sel in ((\"A_main\", 2000, np.arange(len(batch))),\n                         (\"A_sub\", 1800, np.where(batch == 1)[0])):\n        cols = np.sort(var_order[:P])\n        sel = np.asarray(sel)\n        out[name] = dict(X=Z[np.ix_(sel, cols)].astype(np.float64),\n                         batch=batch[sel], status=status[sel],\n                         age=age[sel], gender=gender[sel],\n                         probe_ids=[shared[c] for c in cols])\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Families B/C: tabular designs + random Fourier featurization\n# ---------------------------------------------------------------------------\n\n\ndef rff_map(Xs: np.ndarray, P: int, seed: int) -> tuple[np.ndarray, dict]:\n    \"\"\"Deterministic RFF map with median-heuristic bandwidth.\"\"\"\n    rng = np.random.default_rng(seed)\n    sub = Xs[rng.choice(len(Xs), size=min(500, len(Xs)), replace=False)]\n    D2 = (np.sum(sub ** 2, 1)[:, None] + np.sum(sub ** 2, 1)[None, :]\n          - 2.0 * sub @ sub.T)\n    med = float(np.median(D2[np.triu_indices_from(D2, 1)]))\n    h = float(np.sqrt(max(med / 2.0, 1e-12)))\n    W = rng.standard_normal((Xs.shape[1], P)) / h\n    b = rng.uniform(0.0, 2.0 * np.pi, size=P)\n    Z = np.cos(Xs @ W + b) * np.sqrt(2.0 / P)\n    # frozen convention: RFF block is z-scored per pooled column so the\n    # design obeys the model-card normalization sigma_u ~ O(1); this\n    # rescales the whole spectrum affinely and leaves its shape intact.\n    mu_z, sd_z = Z.mean(axis=0, keepdims=True), Z.std(axis=0, keepdims=True)\n    Z = (Z - mu_z) / np.maximum(sd_z, 1e-12)\n    return Z.astype(np.float64), {\"h\": round(h, 6), \"P\": P, \"seed\": seed}\n\n\ndef build_tabular():\n    out = {}\n    ctrl_names = [\"inc\", \"incsq\", \"agesq\", \"age\", \"male\", \"marr\", \"fsize\"]\n    # each family is optional at build time so shards can fetch only their\n    # own primary sources (Colab self-containment)\n    if (OUT / \"k401ksubs.csv\").exists():\n        with open(OUT / \"k401ksubs.csv\") as fh:\n            rows = list(csv.DictReader(fh))\n        D_raw = np.array([[float(r[\"e401k\"]), float(r[\"nettfa\"])]\n                          for r in rows])\n        C = np.array([[float(r[c]) for c in ctrl_names] for r in rows])\n        rng = np.random.default_rng(GLOBAL_SEED + 77)\n        sub = np.sort(rng.choice(len(C), size=800, replace=False))\n        Cs = C[sub]\n        mu, sd = Cs.mean(0), Cs.std(0)\n        Cs_std = (Cs - mu) / np.maximum(sd, 1e-12)\n        for name, P in ((\"C_main\", 1600), (\"C_wide\", 160)):\n            Z, info = rff_map(Cs_std, P, GLOBAL_SEED + 78)\n            out[name] = dict(X=Z, treat=D_raw[sub, 0],\n                             outcome=D_raw[sub, 1], ctrl_names=ctrl_names,\n                             rff=info, raw_controls=Cs_std)\n    if (OUT / \"ihdp.csv\").exists():\n        with open(OUT / \"ihdp.csv\") as fh:\n            rows = list(csv.DictReader(fh))\n        treat = np.array([float(r[\"treatment\"]) for r in rows])\n        yobs = np.array([float(r[\"outcome\"]) for r in rows])\n        H = np.array([[float(r[f\"feature{k}\"]) for k in range(25)]\n                      for r in rows])\n        mu, sd = H.mean(0), H.std(0)\n        Hs = (H - mu) / np.maximum(sd, 1e-12)\n        for name, P in ((\"B_main\", 750), (\"B_wide\", 150)):\n            Z, info = rff_map(Hs, P, GLOBAL_SEED + 88)\n            out[name] = dict(X=Z, treat=treat, outcome=yobs, rff=info,\n                             raw_controls=Hs)\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Spectral audit shared by all families\n# ---------------------------------------------------------------------------\n\n\ndef spectral_audit(X: np.ndarray) -> dict:\n    sys.path.insert(0, str(ROOT / \"code\"))\n    from de_formulas import (estimate_noise_scales, onatski_select,\n                             tw_mu_sigma, tw_threshold)\n    n, p = X.shape\n    Xc = X - X.mean(axis=0, keepdims=True)\n    if p <= n:\n        d = np.linalg.eigvalsh(Xc.T @ Xc / n)[::-1]\n    else:\n        d = np.maximum(np.linalg.eigvalsh(Xc @ Xc.T / n)[::-1], 0.0)\n    c = p / n\n    # FROZEN benchmark noise-floor convention (docs/benchmark_protocol.md):\n    # gene-expression designs show an approximate white bulk and the shared\n    # estimator applies; kernel-featurized designs decay smoothly and admit\n    # NO MP-consistent bulk. Uniform rule for all benchmarks:\n    #     se2_bench = max( q25(d), 1e-3 * mean(d) ),\n    # and the exact algebraic decomposition Sigma_X = se2 I + sum_j l_j se2\n    # q_j q_j' gives l_hat_j = d_j / se2 - 1 >= 0 with no BBP inversion step.\n    se2_mp, sy2 = estimate_noise_scales(d, c)\n    se2 = float(max(np.quantile(d, 0.25), 1e-3 * float(np.mean(d))))\n    mu_np, sig_np = tw_mu_sigma(n, p)\n    lam_max = float(d[0])\n    ktop = int(min(max(onatski_select(d), 1), 10))\n    return {\n        \"n\": int(n), \"p\": int(p), \"c\": round(c, 3),\n        \"se2_mp_bulk_est\": round(float(se2_mp), 6),\n        \"se2_bench\": round(float(se2), 6),\n        \"sigma_y2_hat\": round(float(sy2), 4),\n        \"lam_max_cov\": round(lam_max, 3),\n        \"tw_stat_lammax\": round(float((lam_max * n - mu_np) / sig_np), 1),\n        \"outlier99_white\": bool(lam_max > tw_threshold(n, p, 1.0)),\n        \"r_hat_onatski\": int(onatski_select(d)),\n        \"ktop_alarm\": ktop,\n        \"r_inj\": ktop,\n        \"tau_top10\": [round(float(v), 2) for v in (d[:10] / se2)],\n        \"l_hat_top10\": [round(float(v), 2) for v in (d[:10] / se2 - 1.0)],\n        \"lambda_col_sd_top10\": [round(float(np.sqrt(max(v, 0.0))), 3)\n                                for v in (d[:10] - se2)],\n    }\n\n\ndef known_results() -> dict:\n    \"\"\"Empirical reproduction of each family's canonical result (raw blocks,\n    no injection): the audit trail required by WP 3.2 / plan Section 9.7.\"\"\"\n    import csv\n    res = {}\n    with open(OUT / \"k401ksubs.csv\") as fh:\n        rows = list(csv.DictReader(fh))\n    ctrl = [\"inc\", \"incsq\", \"agesq\", \"age\", \"male\", \"marr\", \"fsize\"]\n    D = np.array([float(r[\"e401k\"]) for r in rows])\n    Y = np.array([float(r[\"nettfa\"]) for r in rows])\n    Cm = np.array([[float(r[c]) for c in ctrl] for r in rows])\n    M = np.column_stack([np.ones(len(rows)), D, Cm])\n    coef, *_ = np.linalg.lstsq(M, Y, rcond=None)\n    res[\"k401k_ols_e401k_full\"] = {\n        \"coef\": round(float(coef[1]), 3),\n        \"mean_outcome\": round(float(Y.mean()), 2),\n        \"n\": len(rows),\n        \"claim\": \"401(k) ELIGIBILITY is associated with higher net financial \"\n                 \"assets (positive coefficient), the Poterba-Venti-Wise \"\n                 \"finding reproduced by plain covariate adjustment\",\n    }\n    with open(OUT / \"ihdp.csv\") as fh:\n        rows = list(csv.DictReader(fh))\n    t = np.array([float(r[\"treatment\"]) for r in rows])\n    y = np.array([float(r[\"outcome\"]) for r in rows])\n    H = np.array([[float(r[f\"feature{k}\"]) for k in range(25)] for r in rows])\n    diff = float(y[t == 1].mean() - y[t == 0].mean())\n    M = np.column_stack([np.ones(len(rows)), t, H])\n    coef, *_ = np.linalg.lstsq(M, y, rcond=None)\n    res[\"ihdp_naive\"] = {\n        \"unadjusted_diff\": round(diff, 3),\n        \"ols_adjusted_coef\": round(float(coef[1]), 3),\n        \"experimental_benchmark_ate\": 4.0,\n        \"claim\": \"IHDP point estimates land at the experimental-benchmark \"\n                 \"scale (ATE ~ 4.0, Hill 2011): covariate adjustment gives \"\n                 \"3.93, slightly below it; stored as this mirror's \"\n                 \"reproduction anchor\",\n    }\n    return res\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    audit = {}\n    for name, payload in {**build_addneuromed(), **build_tabular()}.items():\n        prof = spectral_audit(payload[\"X\"])\n        audit[name] = prof\n        keep = {k: v for k, v in payload.items() if isinstance(v, np.ndarray)}\n        meta = {k: v for k, v in payload.items() if k not in keep}\n        np.savez_compressed(OUT / f\"{name}.npz\", **keep,\n                            config_name=name, meta_json=json.dumps(\n                                {**meta, \"profile\": prof}, default=str))\n        print(name, json.dumps(prof))\n    (OUT / \"spectral_audit.json\").write_text(json.dumps(\n        {\"spectral_profiles\": audit, \"known_results\": known_results()},\n        indent=1))\n    print(\"wrote\", OUT / \"spectral_audit.json\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'benchmarks_data.py').write_text(harness_benchmarks_data)
assert hashlib.sha256((p/'benchmarks_data.py').read_bytes()).hexdigest()[:16] == 'a4bd80f13fe03ca6', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('benchmarks_data.py ok')

### harness: `benchmarks.py` (sha256 97cd55c84440ece4)

In [ ]:
harness_benchmarks = "\"\"\"SCF Phase 3 benchmark machinery (WP 3.2/3.3): semi-synthetic injection on\nreal designs, calibrated alarm + mandatory baselines, arms runner with\ncheckpoint/resume, and the F12-law frontier prediction g* per benchmark.\n\nFrozen spec: configs/benchmarks_frozen.yaml (v1). Two-pass order is binding:\npass 1 = null + perm_null arms only (thresholds and g* into\nresults/benchmark_freeze.json); pass 2 = everything else. All randomness\nflows from SeedSequence([GLOBAL_SEED, cell_hash, rep]) so matched-null twins\nshare beta/f/eps draws with positive arms at equal rep index.\n\nInjection model (identical semantics to simulator.gen_data M1/M2):\n    X_obs = Xc_base + f @ Lam'\n    Y     = X_obs @ beta + f @ gam + eps\nso Cov(X_obs, Y) = Sigma_obs beta + Lambda gamma with\nSigma_obs = Sigma_base + Lambda Lambda', exactly the M1 algebra; the ground\ntruth is beta wrt the OBSERVED design. Null twins drop f from BOTH blocks.\n\nBaselines implemented here (both flagged APPROXIMATE transcriptions, same\npolicy as Phase 2's ucm_strength):\n  ucm_rho  : response-aware confounding-variance share proxy in the spirit\n             of Rendsburg et al. (2022): sum_j g2hat_j l_j/(1+l_j) / mean(d)\n             over the alarm coordinates, g2hat from estimators.estimate_gamma2.\n  js_asym  : Janzing-Schoelkopf-style spectral asymmetry (2018): eigenvalue\n             drops of the design covariance after removing the rank-one\n             response-explained component, max relative drop over the top-K\n             interlacing roots.\nBoth are calibrated by the permuted-Y null arm (frozen q95 thresholds), not\nby their papers' asymptotics.\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfor _v in (\"OMP_NUM_THREADS\", \"OPENBLAS_NUM_THREADS\", \"MKL_NUM_THREADS\",\n           \"NUMEXPR_NUM_THREADS\", \"VECLIB_MAXIMUM_THREADS\"):\n    import os\n\n    os.environ.setdefault(_v, \"1\")\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / \"code\"))\n\nimport pandas as pd  # noqa: E402\n\nfrom de_formulas import (  # noqa: E402\n    bbp_location,\n    ledger_hash,\n    minnorm_capture,\n    mp_edges,\n    onatski_select,\n    tw_mu_sigma,\n    tw_threshold,\n)\nfrom detection import compute_stats, rejections  # noqa: E402\nfrom estimators import estimate_gamma2  # noqa: E402\nfrom simulator import GLOBAL_SEED, spectrum  # noqa: E402\n\nBENCH_DIR = ROOT / \"data\" / \"benchmarks\"\nRAW_DIR = BENCH_DIR / \"raw\"\nFREEZE_PATH = ROOT / \"results\" / \"benchmark_freeze.json\"\n\n\n# ---------------------------------------------------------------------------\n# benchmark objects\n# ---------------------------------------------------------------------------\n\n\nclass Bench:\n    \"\"\"One processed real design plus its frozen spectral profile.\"\"\"\n\n    def __init__(self, name: str):\n        self.name = name\n        z = np.load(BENCH_DIR / f\"{name}.npz\", allow_pickle=True)\n        self.X = np.ascontiguousarray(z[\"X\"], dtype=np.float64)\n        meta = json.loads(str(z[\"meta_json\"]))\n        self.meta = meta\n        self.n, self.p = self.X.shape\n        self.c = self.p / self.n\n        Xc = self.X - self.X.mean(axis=0, keepdims=True)\n        self.Xc = Xc\n        self.eig = spectrum(Xc)\n        d = self.eig[0]\n        self.d = d\n        self.se2 = float(max(np.quantile(d, 0.25),\n                             1e-3 * float(np.mean(d))))\n        self.ktop = int(min(max(onatski_select(d), 1), 10))\n        self.r_inj = self.ktop\n        self.l_hat = np.maximum(d[:10] / self.se2 - 1.0, 0.0)\n        self.V = self.eig[1]\n\n    def lam_matrix(self, r: int) -> np.ndarray:\n        r = int(max(1, min(r, 10)))\n        sds = np.sqrt(np.maximum(self.d[:r] - self.se2, 1e-12))\n        return self.V[:, :r] * sds[None, :]\n\n\ndef cid_for(config_name: str, arm: str) -> str:\n    import hashlib\n\n    payload = f\"benchmarks_frozen_v1|{config_name}|{arm}\"\n    return hashlib.sha256(payload.encode()).hexdigest()[:12]\n\n\ndef rng_for(config_name: str, arm: str, rep: int) -> np.random.Generator:\n    return np.random.default_rng(\n        np.random.SeedSequence(\n            entropy=[GLOBAL_SEED, int(cid_for(config_name, arm)[:8], 16), rep]\n        )\n    )\n\n\ndef gamma_dir(bench: Bench, kind: str, r: int) -> np.ndarray:\n    dirv = np.zeros(max(r, bench.r_inj))\n    if kind == \"spread\":\n        dirv[:r] = 1.0 / np.sqrt(max(r, 1))\n    elif kind == \"top\":\n        dirv[0] = 1.0\n    elif kind == \"weak\":\n        dirv[r - 1] = 1.0\n    else:\n        raise ValueError(kind)\n    return dirv[:r]\n\n\n# ---------------------------------------------------------------------------\n# one replication\n# ---------------------------------------------------------------------------\n\n\ndef _draw_core(bench: Bench, config_name: str, arm: str, rep: int,\n               r: int):\n    rng = rng_for(config_name, arm, rep)\n    beta = rng.standard_normal(bench.p)\n    beta /= float(np.linalg.norm(beta))\n    f = rng.standard_normal((bench.n, r))\n    eps = rng.standard_normal(bench.n)\n    pi = None\n    delta = None\n    nu = None\n    if bench.meta.get(\"m2\"):\n        kpi = max(3, bench.p // 100)\n        pi = np.zeros(bench.p)\n        pi[:kpi] = 1.0 / np.sqrt(kpi)\n        delta = rng.standard_normal(r)\n        delta *= 0.3 / max(float(np.linalg.norm(delta)), 1e-12)\n        nu = rng.standard_normal(bench.n)\n    return dict(beta=beta, f=f, eps=eps, pi=pi, delta=delta, nu=nu)\n\n\ndef run_rep(bench: Bench, config_name: str, arm_spec: dict, arm: str,\n            rep: int, g_star: float | None,\n            coord_scales: np.ndarray | None = None):\n    \"\"\"Return (rows, mean_diff_ols or None).\"\"\"\n    t0 = time.perf_counter()\n    kind = arm_spec[\"type\"]\n    r = bench.r_inj + int(arm_spec.get(\"r_delta\", 0))\n    r = int(min(max(r, 1), 10)) if kind != \"split_half_null\" else bench.r_inj\n    core = _draw_core(bench, config_name, arm, rep, r)\n    beta, f, eps = core[\"beta\"], core[\"f\"], core[\"eps\"]\n    hetero = bool(arm_spec.get(\"hetero_eps\", False))\n    if hetero:\n        w = rng_for(config_name, arm + \"_het\", rep).chisquare(1.0, bench.n)\n        eps = eps * np.sqrt(w / w.mean())\n    gscale = float(arm_spec.get(\"g_scale\", 1.0))\n    gam_scale = 0.0 if arm_spec.get(\"gamma_zero\") else (\n        (g_star if g_star is not None else 1.0) * gscale)\n    dirv = gamma_dir(bench, arm_spec.get(\"dir\", \"spread\"), r)\n    gam = gam_scale * dirv\n\n    rows = []\n    mean_diff = None\n\n    def stats_block(Xc_used, Y_used, eig_used, tag_rows):\n        det = compute_stats(Xc_used, Y_used - Y_used.mean(), eig_used)\n        zeta = raw_z_coords(Xc_used, Y_used - Y_used.mean(), eig_used,\n                            bench.ktop)\n        t_alarm, k_used = bench_alarm_stat(Xc_used, Y_used - Y_used.mean(),\n                                           eig_used, bench, coord_scales)\n        row = {\n            \"config_id\": cid_for(config_name, arm),\n            \"config_name\": config_name, \"arm\": arm, \"rep\": rep,\n            **det, **rejections(det),\n            \"T_bench\": t_alarm,\n            \"ktop_bench\": k_used,\n            \"ucm_rho\": ucm_rho(Xc_used, Y_used - Y_used.mean(), eig_used,\n                               bench),\n            \"js_asym\": js_asym_from_stats(eig_used, Xc_used,\n                                          Y_used - Y_used.mean()),\n        }\n        for j, zv in enumerate(zeta):\n            row[f\"zeta{j}\"] = float(zv)\n        tag_rows.append(row)\n        return row\n\n    if kind == \"bench\":\n        Lam = bench.lam_matrix(r)\n        Xobs = bench.Xc + f @ Lam.T\n        Y = Xobs @ beta + f @ gam + eps\n        eig_obs = spectrum(Xobs)\n        Xc_obs = Xobs - Xobs.mean(axis=0, keepdims=True)\n        row = stats_block(Xc_obs, Y, eig_obs, rows)\n        b_ols = _ols_beta(Xc_obs, Y - Y.mean(), eig_obs)\n        mean_diff = b_ols - beta\n        row[\"rel_bias_dir\"] = float(np.linalg.norm(mean_diff))\n    elif kind == \"permute_y_of_null\":\n        Xnull = bench.Xc\n        Y0 = Xnull @ beta + eps\n        prng = rng_for(config_name, arm + \"_p\", rep)\n        Yperm = prng.permutation(Y0)\n        stats_block(Xnull, Yperm, bench.eig, rows)\n    elif kind == \"split_half_null\":\n        prng = rng_for(config_name, arm + \"_s\", rep)\n        idx = np.sort(prng.choice(bench.n, size=bench.n // 2, replace=False))\n        Xh = bench.Xc[idx]\n        Yh = Xh @ beta + eps[idx]\n        Xh = Xh - Xh.mean(axis=0, keepdims=True)\n        eig_h = spectrum(Xh)\n        stats_block(Xh, Yh, eig_h, rows)\n    elif kind == \"m2\":\n        Lam = bench.lam_matrix(r)\n        Xobs = bench.Xc + f @ Lam.T\n        D = Xobs @ core[\"pi\"] + f @ core[\"delta\"] + core[\"nu\"]\n        Y = 1.0 * D + Xobs @ beta + f @ gam + eps\n        eig_obs = spectrum(Xobs)\n        Xc_obs = Xobs - Xobs.mean(axis=0, keepdims=True)\n        Dc = D - D.mean()\n        Yc = Y - Y.mean()\n        taus = tau_estimators(Xc_obs, Dc, Yc, eig_obs, bench)\n        row = stats_block(Xc_obs, Y, eig_obs, rows)\n        row.update(taus)\n    else:\n        raise ValueError(kind)\n\n    for row in rows:\n        row[\"runtime_s\"] = time.perf_counter() - t0\n    return rows, mean_diff\n\n\ndef _ols_beta(Xc, Yc, eig):\n    from simulator import fit_ols\n\n    return fit_ols(Xc, Yc, eig)\n\n\ndef ucm_rho(Xc: np.ndarray, Yc: np.ndarray, eig, bench: Bench) -> float:\n    \"\"\"Response-aware confounding-variance share proxy (APPROXIMATE UCM).\n\n    rho_hat = sum_{j<ktop} g2hat_j l_j/(1+l_j) / mean(d): the estimated\n    response-linked variance carried by the leading directions, divided by\n    the total design variance share of the bulk. Monotone in the injected\n    link by construction; calibration is permutation-based upstream.\"\"\"\n    n, p = Xc.shape\n    c = p / n\n    k = min(bench.ktop, len(bench.l_hat))\n    l_hat = bench.l_hat[:k]\n    g2 = estimate_gamma2(Xc, Yc, eig, l_hat, c)\n    conf_var = float(np.sum(g2 * l_hat / (1.0 + l_hat)))\n    return float(conf_var / max(float(np.mean(eig[0])), 1e-12))\n\n\ndef raw_z_coords(Xc: np.ndarray, Yc: np.ndarray, eig,\n                 ktop: int) -> np.ndarray:\n    \"\"\"Raw cross-moment spike coordinates zeta_j = sqrt(n) v_j'b / sqrt(d_j).\n\n    b = Xc'Yc/n with Yc CENTERED but NOT standardized: the coordinate mean\n    shift under H1 is v_j'Lambda gamma (absolute units), free of any\n    response-scale dilution.\"\"\"\n    n = len(Yc)\n    d, V = eig\n    k = min(max(int(ktop), 1), len(d))\n    b = Xc.T @ Yc / n\n    return np.sqrt(n) * (V[:, :k].T @ b) / np.sqrt(d[:k])\n\n\ndef bench_alarm_stat(Xc: np.ndarray, Yc: np.ndarray, eig, bench: Bench,\n                     coord_scales: np.ndarray | None = None):\n    \"\"\"Phase-3 gate alarm (deviation D-B0, pre-pass-1 freeze).\n\n    T = max_{j < ktop} |zeta_j| / s_j with s_j the PER-COORDINATE EMPIRICAL\n    NULL SCALES s_j = sqrt(mean_null(zeta_j^2)) estimated from the matched\n    twins (pass 1) and frozen before any positive arm runs. Rationale: real\n    benchmark geometries violate the MP-white-bulk premise of the F12 law\n    (family A's smooth bulk, huge spikes), so analytic coordinate variances\n    misestimate by geometry-dependent constants; the twin-estimated scales\n    absorb every such factor by construction. Before scales are available\n    (inside pass 1 itself) s_j = 1 is used and only the pooled mc95 of T\n    matters for thresholding; scale estimation consumes the SAME null pool\n    (in-sample for size, out-of-sample for every control/positive arm).\n\n    Returns (T, ktop_used).\"\"\"\n    zeta = raw_z_coords(Xc, Yc, eig, bench.ktop)\n    if coord_scales is None:\n        s = np.ones(len(zeta))\n    else:\n        s = np.maximum(np.asarray(coord_scales, float), 1e-12)\n    return float(np.max(np.abs(zeta) / s)), int(len(zeta))\n\n\ndef js_asym_from_stats(eig, Xc: np.ndarray, Yc: np.ndarray,\n                       K: int | None = None) -> float:\n    \"\"\"Janzing-Schoelkopf-style asymmetry (APPROXIMATE transcription).\n\n    C_R = C - m m'/var(Y) removes the response-explained rank-one component;\n    its top eigenvalues interlace with d_i and the relative drops\n    (d_i - lam_i)/d_i measure how concentrated the response-explained\n    variation is on dominant design directions. Statistic = max drop over\n    i < K (K = ktop + 2). Calibrated by permutation upstream.\"\"\"\n    d, V = eig\n    n = len(Yc)\n    m = Xc.T @ Yc / n\n    sy = float(np.mean(Yc ** 2))\n    if sy <= 0:\n        return 0.0\n    w = V.T @ m\n    K = K or 4\n    K = int(min(K, len(d) - 1))\n\n    def h(lam):\n        return float(np.sum(w ** 2 / (sy * (d - lam))))\n\n    drops = []\n    for i in range(K):\n        lo, hi = d[i + 1], d[i]\n        if hi - lo < 1e-14 * max(hi, 1e-300):\n            continue\n        # h is strictly increasing on (lo, hi) with h(hi^-) -> +inf iff\n        # w_i != 0; a root of h(lam) = 1 exists iff h just below hi > 1.\n        eps_hi = 1e-9 * (hi - lo)\n        if h(hi - eps_hi) <= 1.0:\n            continue\n        a_, b_ = lo + eps_hi, hi - eps_hi\n        for _ in range(80):\n            mid = 0.5 * (a_ + b_)\n            if h(mid) > 1.0:\n                b_ = mid\n            else:\n                a_ = mid\n            if b_ - a_ < 1e-11 * hi:\n                break\n        lam = 0.5 * (a_ + b_)\n        drops.append((d[i] - lam) / d[i])\n    return float(max(drops)) if drops else 0.0\n\n\ndef tau_estimators(Xc, Dc, Yc, eig, bench: Bench) -> dict:\n    from de_formulas import onatski_select as _on\n\n    d, V = eig\n\n    def tau_joint(A):\n        coef, *_ = np.linalg.lstsq(A, Yc, rcond=None)\n        return float(coef[0])\n\n    out = {}\n    out[\"tau_ols\"] = tau_joint(np.column_stack([Dc, Xc]))\n    k = int(max(_on(d), bench.r_inj))\n    S = Xc @ V[:, :k]\n    out[\"tau_trim_onatski\"] = tau_joint(np.column_stack([Dc, S]))\n    lam = 1.0\n    n = len(Dc)\n    Sc = Xc.T @ Xc / n + lam * np.eye(Xc.shape[1])\n    rhs = Xc.T @ Yc / n\n    c_vec = Xc.T @ Dc / n\n    m = float(Dc @ Dc) / n\n    Scinv_c = np.linalg.solve(Sc, c_vec)\n    Scinv_rhs = np.linalg.solve(Sc, rhs)\n    out[\"tau_ridge1\"] = (\n        float(Dc @ Yc) / n - c_vec @ Scinv_rhs\n    ) / (m - c_vec @ Scinv_c)\n    out[\"k_trim\"] = k\n    return out\n\n\n# ---------------------------------------------------------------------------\n# frontier prediction g* (F12 law; mirrors phase2_analysis construction)\n# ---------------------------------------------------------------------------\n\n\ndef predicted_g_star(bench: Bench, mc95: float,\n                     coord_scales: np.ndarray | None = None,\n                     r: int | None = None, seed: int = 0) -> float:\n    \"\"\"F12-law frontier for the benchmark alarm (deviation D-B0 form).\n\n    Per-unit-g mean shift of the raw coordinate zeta_j under H1:\n        m_j(g) = g * dir_j * omega_j * sqrt(n) * sqrt(se2 l_j) / sqrt(d_pred)\n    with d_pred = bbp_location(l_j, c, se2) and omega the clipped\n    min-norm-capture weight at c > 1. Coordinate noise scales are the\n    EMPIRICAL twin scales s_j (D-B0), so the standardized shift is\n    m_j(g)/s_j and the max-statistic power curve follows the same MC\n    construction as phase2_analysis.predicted_frontier_g.\n    \"\"\"\n    c, n = bench.c, bench.n\n    r = int(r if r is not None else bench.r_inj)\n    ktop_eff = max(bench.ktop, r)\n    sup = list(range(min(r, ktop_eff)))\n    if not sup:\n        return float(\"inf\")\n    dirv = np.ones(len(sup)) / np.sqrt(len(sup))\n    if coord_scales is None:\n        s = np.ones(len(sup))\n    else:\n        s = np.maximum(np.asarray(coord_scales, float)[:len(sup)], 1e-12)\n    slope = np.zeros(len(sup))\n    for i, j in enumerate(sup):\n        lj = float(bench.l_hat[j])\n        dj = bbp_location(lj, c, bench.se2)\n        omega = float(np.clip(minnorm_capture(np.array([lj]), c)[0], 0, 1)) \\\n            if c > 1 else 1.0\n        slope[i] = (np.sqrt(n) * omega * np.sqrt(bench.se2 * lj) * dirv[i] /\n                    (np.sqrt(dj) * s[i]))\n    if np.max(slope) <= 0:\n        return float(\"inf\")\n    rr = np.random.default_rng(seed)\n    z0 = np.abs(rr.normal(size=(20000, len(sup))))\n    thr_sim = float(np.quantile(z0.max(axis=1), 0.95))\n    scale = mc95 / thr_sim\n    for g in np.linspace(0.01, 20.0, 400):\n        z1 = np.abs(rr.normal(size=(20000, len(sup))) +\n                    slope[None, :] * g).max(axis=1)\n        if float((z1 * scale > mc95).mean()) >= 0.8:\n            return round(float(g), 3)\n    return float(\"inf\")\n\n\n# ---------------------------------------------------------------------------\n# arms / cells runner (checkpoint-resume like runners.run_cell)\n# ---------------------------------------------------------------------------\n\nARMS_ORDER = [\"null\", \"perm_null\", \"pos_half\", \"pos_1\", \"pos_2\",\n              \"splithalf\", \"align_top\", \"align_weak\", \"rinj_minus\",\n              \"rinj_plus\", \"hetero_eps\", \"m2_null\", \"m2_pos\"]\n\nPASS1_ARMS = [\"null\", \"perm_null\"]\n\n\ndef arm_specs():\n    import yaml\n\n    cfg = yaml.safe_load((ROOT / \"configs\" / \"benchmarks_frozen.yaml\")\n                         .read_text())\n    return cfg\n\n\ndef skip_arm(bench: Bench, arm: str, spec: dict) -> bool:\n    if spec.get(\"only\") and bench.name not in spec[\"only\"]:\n        return True\n    thr = spec.get(\"skip_if_r_inj_leq\")\n    if thr is not None and bench.r_inj <= int(thr):\n        return True\n    return False\n\n\ndef _atomic_write_parquet(df: pd.DataFrame, path: Path):\n    tmp = path.with_suffix(\".tmp.parquet\")\n    df.to_parquet(tmp, index=False)\n    tmp.replace(path)\n\n\ndef run_cell(job: dict):\n    arm = job[\"arm\"]\n    spec = job[\"spec\"]\n    raw_path = RAW_DIR / job[\"config_name\"] / f\"{arm}.parquet\"\n    means_path = RAW_DIR / job[\"config_name\"] / f\"{arm}_means.npz\"\n    start_rep = 0\n    prev_frames = []\n    if raw_path.exists():\n        try:\n            have = pd.read_parquet(raw_path, columns=[\"rep\"])\n            n_have = int(have[\"rep\"].nunique())\n            if n_have >= spec[\"reps\"]:\n                print(f\"[skip] {job['config_name']}/{arm} done ({n_have} reps)\",\n                      flush=True)\n                return arm, 0.0\n        except Exception:\n            pass\n    bench = Bench(job[\"config_name\"])\n    raw_path.parent.mkdir(parents=True, exist_ok=True)\n    if raw_path.exists():\n        try:\n            have = pd.read_parquet(raw_path, columns=[\"rep\"])\n            n_have = int(have[\"rep\"].nunique())\n            if n_have >= spec[\"reps\"]:\n                return arm, 0.0\n            start_rep = n_have\n            prev_frames = [pd.read_parquet(raw_path)]\n        except Exception:\n            start_rep = 0\n    t0 = time.perf_counter()\n    frames = list(prev_frames)\n    acc_sum = np.zeros(bench.p)\n    acc_n = 0\n    g_star = job.get(\"g_star\")\n    cs = job.get(\"coord_scales\")\n    since_flush = 0\n    if start_rep and means_path.exists():\n        try:\n            mz = np.load(means_path)\n            acc_sum[: len(mz[\"mean_bias\"])] += (\n                np.asarray(mz[\"mean_bias\"], float) * float(mz[\"n_reps\"]))\n            acc_n = int(mz[\"n_reps\"])\n        except Exception:\n            acc_n = 0\n    for rep in range(start_rep, spec[\"reps\"]):\n        rows, mean_diff = run_rep(bench, bench.name, spec, arm, rep, g_star,\n                                  coord_scales=cs)\n        frames.append(pd.DataFrame(rows))\n        if mean_diff is not None:\n            acc_sum += mean_diff\n            acc_n += 1\n        since_flush += 1\n        if since_flush >= 20:\n            _atomic_write_parquet(pd.concat(frames, ignore_index=True),\n                                  raw_path)\n            if acc_n:\n                np.savez_compressed(means_path,\n                                    mean_bias=(acc_sum / acc_n),\n                                    n_reps=acc_n)\n            since_flush = 0\n    df = pd.concat(frames, ignore_index=True)\n    _atomic_write_parquet(df, raw_path)\n    if acc_n:\n        np.savez_compressed(means_path,\n                            mean_bias=(acc_sum / acc_n).astype(np.float64),\n                            n_reps=acc_n)\n    dt = time.perf_counter() - t0\n    print(f\"[done] {bench.name}/{arm}: {len(df)} rows (+{spec['reps'] - start_rep}) \"\n          f\"in {dt:.1f}s\", flush=True)\n    return f\"{bench.name}/{arm}\", dt\n\n\ndef build_jobs(pass_name: str, freeze: dict | None, workers: int = 3):\n    cfg = arm_specs()\n    audit = json.loads((BENCH_DIR / \"spectral_audit.json\").read_text())\n    profiles = audit.get(\"spectral_profiles\", audit)\n    jobs = []\n    for name, ccfg in cfg[\"configs\"].items():\n        r_inj = int(profiles[name][\"r_inj\"])\n        for arm in ARMS_ORDER:\n            spec = cfg[\"arms\"][arm]\n            thr = spec.get(\"skip_if_r_inj_leq\")\n            if spec.get(\"only\") and name not in spec[\"only\"]:\n                continue\n            if thr is not None and r_inj <= int(thr):\n                continue\n            want = PASS1_ARMS if pass_name == \"pass1\" else \\\n                [a for a in ARMS_ORDER if a not in PASS1_ARMS]\n            if arm not in want:\n                continue\n            gs = None\n            cs = None\n            if pass_name == \"pass2\" and freeze:\n                entry = freeze[\"configs\"][name]\n                gs = entry[\"g_star\"]\n                cs = np.asarray(entry[\"coord_scales\"], float)\n            jobs.append({\"config_name\": name, \"arm\": arm, \"spec\": spec,\n                         \"g_star\": gs, \"coord_scales\": cs})\n    return jobs\n\n\ndef run_jobs(jobs: list[dict], workers: int = 3):\n    from multiprocessing import Pool\n\n    if workers <= 1:\n        results = [run_cell(j) for j in jobs]\n    else:\n        with Pool(workers) as pool:\n            results = pool.map(run_cell, jobs)\n    return results\n"
import pathlib, hashlib
p = pathlib.Path('code'); p.mkdir(exist_ok=True)
(p / 'benchmarks.py').write_text(harness_benchmarks)
assert hashlib.sha256((p/'benchmarks.py').read_bytes()).hexdigest()[:16] == '97cd55c84440ece4', 'harness hash mismatch'
import sys; sys.path.insert(0, 'code')
print('benchmarks.py ok')

### embedded frozen benchmark configuration

In [ ]:
cfg_yaml = "# SCF Phase 3 frozen benchmark configuration (WP 3.2).\n# FROZEN 2026-08-24 BEFORE any comparative result was run. Any change after\n# data generation requires a deviation entry (D-B1, D-B2, ...) in\n# docs/benchmark_protocol.md and a version bump of this file.\n#\n# Two-pass order (binding):\n#   pass 1: null arms only -> per-config MC thresholds (mc95 of t_maxz) and\n#           predicted frontier g* via the F12 law -> results/benchmark_freeze.json\n#   pass 2: positive arms, controls, sensitivity, M2 block, evaluated against\n#           the FROZEN thresholds and g* values.\n# No pass-2 output may be generated before benchmark_freeze.json exists.\n\nversion: \"benchmarks_frozen_v1\"\ndate: \"2026-08-24\"\nledger_hash: \"11b162ac814d\"\nglobal_seed: 20260823\n\ninputs_sha256:\n  A_main.npz: 862903337437c3c580c5ba91730288fa7fdd7117a064e6c1364a67ddaaa300fb\n  A_sub.npz: 15cdb9f659c74edcea693f4b8e97f0ea7c279f9ffaf7f076daee2f13ea12609b\n  B_main.npz: 1dbc7a11b9a68d40e087d01381a50414ac8aab5bd65fe224c3375df9c7bba178\n  B_wide.npz: 4aecf5aa40da4398bf318dd76fe5da37ef6f0d69a8ea63505e5721dff48a9775\n  C_main.npz: 442f3bc852982718b4a190ab07cef1975ac8cc93b7d832b0b9921a309a5c8f80\n  C_wide.npz: 7cdbfaaffd07061fc3b598d976f7180a0c1ea6c40d1b5d55f00cdec7f7903c21\n\nconventions:\n  noise_floor_bench: \"se2 = max(q25(d), 1e-3 * mean(d)); l_j = d_j/se2 - 1 (exact decomposition)\"\n  injection_model: \"X_obs = Xc_base + f @ Lam'; Y = X_obs @ beta + f @ gam + eps; beta Haar ||beta||=1; f ~ N(0,I); eps ~ N(0,1)\"\n  null_twin: \"same seeds with the factor draw removed from BOTH X_obs and Y\"\n  loading_geometry: \"Lam columns = first r_inj sample eigenvectors of the base design, column sd sqrt(d_j - se2)\"\n  default_gamma_dir: \"equal mass over the r_inj injected coordinates\"\n  m2_block: \"D = X_obs pi + f delta + nu; pi sparse kpi=max(3,p//100) coords at 1/sqrt(kpi); ||delta||=0.3; nu~N(0,1); tau_true=1\"\n\nconfigs:\n  A_main: {family: addneuromed, n: 717, p: 2000, c: 2.789, r_inj: 2}\n  A_sub:  {family: addneuromed, n: 388, p: 1800, c: 4.639, r_inj: 1, role: batch_free_sensitivity}\n  B_main: {family: ihdp, n: 747, p: 750, c: 1.004, r_inj: 1}\n  B_wide: {family: ihdp, n: 747, p: 150, c: 0.201, r_inj: 1, role: low_c_sensitivity}\n  C_main: {family: k401k, n: 800, p: 1600, c: 2.000, r_inj: 1, m2: true}\n  C_wide: {family: k401k, n: 800, p: 160, c: 0.200, r_inj: 1, role: low_c_sensitivity}\n\narms:\n  # pass 1 (calibration: thresholds and g*, frozen before any evaluation)\n  \"null\":      {type: bench, gamma_zero: true, reps: 600}\n  perm_null:   {type: permute_y_of_null, reps: 400}\n  # pass 2 (comparative)\n  pos_half:    {type: bench, g_scale: 0.5, reps: 400}\n  pos_1:       {type: bench, g_scale: 1.0, reps: 400}\n  pos_2:       {type: bench, g_scale: 2.0, reps: 400}\n  splithalf:   {type: split_half_null, reps: 300}\n  align_top:   {type: bench, g_scale: 1.0, dir: top, reps: 200, skip_if_r_inj_leq: 1}\n  align_weak:  {type: bench, g_scale: 1.0, dir: weak, reps: 200, skip_if_r_inj_leq: 1}\n  rinj_minus:  {type: bench, g_scale: 1.0, r_delta: -1, reps: 200, skip_if_r_inj_leq: 1}\n  rinj_plus:   {type: bench, g_scale: 1.0, r_delta: +1, reps: 200}\n  hetero_eps:  {type: bench, g_scale: 1.0, hetero_eps: true, reps: 200}\n  m2_null:     {type: m2, gamma_zero: true, reps: 300, only: [C_main]}\n  m2_pos:      {type: m2, g_scale: 1.0, reps: 300, only: [C_main]}\n\nthresholds_and_gates:\n  size_band_negative_controls: [0.02, 0.10]\n  pf1_power_at_2x_gstar_min: 0.80\n  pf1_power_at_0p5x_gstar_max: 0.25\n  alarm_primary: \"S2-bench gate alarm (raw cross-moment zeta coordinates, per-coordinate EMPIRICAL null scales from matched twins + pooled MC-twin max threshold; deviation D-B0 pre-pass-1); frozen gamma-blind S2 and analytic Bonferroni co-reported\"\n  baselines_mandatory: [ucm_strength_permboot, js_asymmetry_permboot, scree_tw99_S0, partial_F_B1]\n  g4_go: \"size in band under negative controls on >= 2 families; positive-control power straddles g*; >=1 predeclared finding survives sensitivity\"\n\ndeviations:\n  D-B0: \"2026-08-24, BEFORE any pass-1 data. Two findings from WP 3.2 smoke testing, both fixed in the benchmark alarm before calibration data: (1) the frozen S2 statistic evaluates the F12 law with a gamma-blind MODEL estimate sigma_y2_hat=mean(d)+se2_hat; under an injected link the true response scale inflates and suppresses the statistic by sigma_y_true/sigma_y_hat (Phase-2 Erratum-3 self-normalization, strong on real geometry). First fix attempt: plug the realized sd(Y)^2 into var_cal (exact cancellation). (2) On family A the MP-white-bulk premise itself fails (smooth bulk, top spike tau ~ 1400 relative to floor), so even realized-scale coordinate variances misestimate by a geometry factor ~ 4 and the null max-statistic is heavy-tailed (mc95 ~ 5.7 vs ~ 3 elsewhere). Final amendment: gate alarm T = max_j |zeta_j|/s_j with zeta_j = sqrt(n) v_j' (Xc'Yc/n)_j / sqrt(d_j) computed on the RAW CENTERED response (H1 mean shift v_j'Lambda gamma enters in absolute units, immune to response-scale dilution) and s_j = twin-estimated per-coordinate null scales sqrt(mean(zeta_j^2)) from the pass-1 matched-null pool, frozen before any positive/control arm runs; pooled mc95 of T completes the threshold. The F12 law survives as the FRONTIER PREDICTION layer with slopes standardized by the same s_j, which is exactly what PF-1 tests. Phase 2 code paths untouched.\"\n\npredeclared_findings:\n  PF-1: \"the calibrated S2 alarm straddles the benchmark-specific predicted frontier on real geometry (size-calibrated under negative controls, power >= 0.8 at 2x g*, <= 0.25 at 0.5x g*)\"\n  PF-2: \"naive spectrum-gazing false-alarms: S0/TW99 inspection rejects on every unmodified real design before any confounding is injected\"\n  PF-3: \"trim-then-regress (Onatski k) recovers tau_true=1 sign/magnitude under 1x-frontier injected confounding on C_main where raw-OLS tau error inflates materially\"\n  PF-4_secondary: \"response-aware UCM strength proxy tracks injected g monotonically but its permuted threshold is invalid as a calibrated confounding test on real geometry\"\n\ngive_up_rules:\n  KILL_1: \"diagnostic miscalibrated on real geometry (null rejection outside [0.02,0.10] across all robust variants)\"\n  KILL_2: \"irreconcilable contradiction with trusted benchmark results, or nothing over {scree, UCM, JS-criterion, F} on all families\"\n  PIVOT_3: \"only one application family works -> restrict applied section to that family\"\n"
import pathlib
pathlib.Path('configs').mkdir(exist_ok=True)
pathlib.Path('configs/benchmarks_frozen.yaml').write_text(cfg_yaml)
print('configs ok')

### acquire primary sources (pinned hashes)

In [ ]:
import urllib.request, hashlib, pathlib

pathlib.Path('data/benchmarks/ihdp.csv').parent.mkdir(parents=True, exist_ok=True)
if not pathlib.Path('data/benchmarks/ihdp.csv').exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/gpeng9/ihdp-causality/master/ihdp.csv', 'data/benchmarks/ihdp.csv')
got = hashlib.sha256(pathlib.Path('data/benchmarks/ihdp.csv').read_bytes()).hexdigest()
assert got == '1c12eb6df2d6a48165b34963e1457a47cfe31fb300c61b5a2c2557ddd84da467', ('source hash mismatch', 'data/benchmarks/ihdp.csv', got)
print('source ok:', 'data/benchmarks/ihdp.csv')

### regenerate + verify the design

In [ ]:

import numpy as np, json, hashlib, sys
sys.path.insert(0, 'harness')
import benchmarks_data as BD

CONFIG = "B_main"
EXPECTED_X_SHA16 = "c263f00d939b3f35"
RAW_DIR_NAME = "B_main"

if CONFIG.startswith('A'):
    fam = BD.build_addneuromed()
else:
    fam = BD.build_tabular()
payload = fam[CONFIG]
X = np.ascontiguousarray(payload['X'], dtype=np.float64)
got = hashlib.sha256(X.tobytes()).hexdigest()[:16]
assert got == EXPECTED_X_SHA16, (got, EXPECTED_X_SHA16)
meta = {k: (v.tolist() if isinstance(v, np.ndarray) else v)
        for k, v in payload.items() if k != 'X'}
meta_json = json.dumps(meta, default=str)
np.savez_compressed('B_main.npz', X=X,
                    config_name=CONFIG, meta_json=meta_json)
print('design verified:', CONFIG, X.shape)


### embedded pass-1 freeze for this config

In [ ]:

freeze_entry = json.loads(r'''{"mc95_S2": 1.9746, "coord_scales": [9.7228], "size_s2_analytic_null": 0.0, "size_s0_null": 1.0, "size_b1_null": 0.7983, "ucm_q95_perm": 0.0, "js_q95_perm": 0.0084, "se2_bench": 0.004366, "ktop_alarm": 1, "r_inj": 1, "l_hat_used": [11048.551], "g_star": 1.012}''')
FREEZE = {"version": "benchmarks_frozen_v1",
          "ledger_hash": "11b162ac814d",
          "configs": {CONFIG: freeze_entry}}
print(json.dumps(freeze_entry, indent=1))


### run all pass-2 arms for this config (checkpointed)

In [ ]:

import shutil, time
from pathlib import Path
import pandas as pd
import benchmarks as B

B.BENCH_DIR = Path('.').resolve()
B.RAW_DIR = B.BENCH_DIR / 'state'
state_cfg = B.RAW_DIR / CONFIG
state_cfg.mkdir(parents=True, exist_ok=True)
dst_npz = B.BENCH_DIR / f'{CONFIG}.npz'
if Path(f'{CONFIG}.npz').resolve() != dst_npz.resolve():
    shutil.copy(f'{CONFIG}.npz', dst_npz)
# spectral audit is only needed by build_jobs; provide it
audit = {"r_inj": freeze_entry["r_inj"]}
(B.BENCH_DIR / 'spectral_audit.json').write_text(json.dumps(
    {"spectral_profiles": {CONFIG: audit}}))

_orig_specs = B.arm_specs
B.arm_specs = lambda: {**_orig_specs(),
                       'configs': {CONFIG: _orig_specs()['configs'][CONFIG]}}
bench = B.Bench(CONFIG)
jobs = B.build_jobs('pass2', FREEZE)
t0 = time.time()
for j in jobs:
    if time.time() - t0 > 5.5 * 3600:
        print('time budget reached; rerun notebook to resume'); break
    B.run_cell(j)
print('shard done or budget-stopped')


### package outputs

In [ ]:

import zipfile, hashlib
zf = zipfile.ZipFile('./scf_bench_B_main.zip', 'w')
manifest = {}
for f in sorted(Path('./state').rglob('*')):
    if f.is_file():
        arc = str(f.relative_to('.'))
        zf.write(f, arcname=arc)
        manifest[arc] = hashlib.sha256(f.read_bytes()).hexdigest()
zf.writestr('manifest.json', json.dumps(manifest, indent=1))
zf.close()
print('wrote scf_bench_B_main.zip')
try:
    from google.colab import files
    files.download('scf_bench_B_main.zip')
    print('Downloaded: scf_bench_B_main.zip')
except Exception as e:
    print('(Not on Colab / download skipped):', e)
